# OBJECTIVES - as to-do list

## Element 1
To provide stats of a thesis's complexity by extracting counts directly from the PDF source for the following metrics:
- [x] number of figures: ``num_figures`` (production ready, unified cell)
- [x] number of tables: ``num_tables`` (production ready, unified cell)
- [] number of equations: ``num_equations``
- [x] number of references: ``num_references``

## Element 2
To provide a measure of the "degree" of scientific grounding metrics ``citation_density``and ``figure_density`` will be calculated alongside an exstraction of the length (in number of pages) of the *litterature review* / *background* section(s):
- [] ``citation_density``(eg. a measure like $num_{citations}/1000 words$)
- [] ``figure_density`` (eg. a measure like $num_{figures}/1000words$)
- [] ``background_length``

## Element 3
An analysis of the linguistics of the thesis, measure could for example be (***suggestions!***):
- [] ``avg_sentence_length`` (number of words in the average sentence)
- [] ``hapax_legomenon_ratio`` (ratio of "words or a form that occurs only once within a particular corpus or body")
- [] others?

# PRODUCTION
Elements 1.1 (`num_figures`), 1.2 (`num_tables`), and 1.4 (`num_references`) are integrated in a unified production flow. Each metric keeps its tuned implementation to preserve current validation performance while still allowing independent future adjustments.


In [1]:
# ==== IMPORTS ====
from pathlib import Path
import re
import fitz
import pandas as pd

## Element 1: `num_figures`, `num_tables` & `num_references` (Production Ready)

### `num_figures` - production

In [8]:
# ==== PRODUCTION PATCH (Atomic): restore legacy num_figures extractor ====
# This override is intentionally metric-specific to recover pre-integration behavior.

def extract_num_figures(pdf_path: str, debug: bool = False) -> int | None:
    """Estimate number of figures with List-of-Figures fast-track and caption fallback."""
    try:
        token_pattern = r"(?:figure|fig\.?|figur|f\s*i\s*g(?:\s*u\s*r(?:\s*e)?)?\.?)"
        arabic_index_pattern = r"(?:[A-Z]\s*[\.-]\s*)?\d+(?:\s*[\.,-]\s*\d+)*(?:\s*[a-zA-Z])?"
        roman_index_pattern = r"[IVXLCDM]{1,7}"
        index_pattern = rf"\(?\s*(?:{arabic_index_pattern}|{roman_index_pattern})\s*\)?"

        caption_start_pattern = re.compile(
            rf"^\s*(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\s*(?P<sep>[:\-\.,])?\s*(?P<tail>.*)$",
            re.IGNORECASE,
        )
        caption_inline_strong_pattern = re.compile(
            rf"(?<!\w)(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\s*(?P<sep>[:\-])\s*(?P<tail>.+)$",
            re.IGNORECASE,
        )
        token_presence_pattern = re.compile(
            rf"(?<!\w){token_pattern}(?!\w)",
            re.IGNORECASE,
        )

        lof_heading_terms = (
            "list of figures",
            "figure list",
            "lof",
            "figures",
            "figurer",
            "figurenliste",
            "figur liste",
            "liste over figurer",
            "figuroversigt",
            "oversigt over figurer",
            "fortegnelse over figurer",
            "liste af figurer",
        )
        numbered_heading_prefix_pattern = rf"(?:{roman_index_pattern}|[A-Z]|\d+(?:\s*[\.-]\s*\d+)*)"
        lof_heading_pattern = re.compile(
            rf"^\s*(?:{numbered_heading_prefix_pattern}\s*[\).:-]?\s+)?(?:{'|'.join(re.escape(term) for term in lof_heading_terms)})\s*:?\s*$",
            re.IGNORECASE,
        )
        lof_entry_pattern = re.compile(
            rf"^\s*(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\b.*$",
            re.IGNORECASE,
        )
        lof_standalone_idx_pattern = re.compile(
            rf"^\s*(?P<idx>{index_pattern})\s*$",
            re.IGNORECASE,
        )
        lof_idx_caption_pattern = re.compile(
            rf"^\s*(?P<idx>{index_pattern})\s+(?P<tail>.+)$",
            re.IGNORECASE,
        )
        caption_letters_pattern = re.compile(r"[A-Za-zÆØÅæøå]")
        generic_heading_pattern = re.compile(
            r"^\s*(chapter\s+\d+|kapitel\s+\d+|references|litteratur|appendix|bilag)\b",
            re.IGNORECASE,
        )
        table_heading_terms = (
            "list of tables",
            "table list",
            "tables",
            "tabeller",
            "liste over tabeller",
            "fortegnelse over tabeller",
            "liste af tabeller",
            "tabeloversigt",
            "oversigt over tabeller",
        )
        table_heading_pattern = re.compile(
            rf"^\s*(?:{numbered_heading_prefix_pattern}\s*[\).:-]?\s+)?(?:{'|'.join(re.escape(term) for term in table_heading_terms)})\s*:?\s*$",
            re.IGNORECASE,
        )

        ignored_heading_markers = (
            "list of figures",
            "figure list",
            "lof",
            "figures",
            "figurer",
            "figurenliste",
            "figur liste",
            "liste over figurer",
            "figuroversigt",
            "oversigt over figurer",
            "fortegnelse over figurer",
            "liste af figurer",
        )
        in_text_cues = (
            "see",
            "shown in",
            "illustrated in",
            "as seen in",
            "som vist i",
            "se",
            "se også",
            "illustreret i",
            "consider",
            "at",
            "on",
            "again",
            ". ",
            "from",
        )
        no_sep_reference_tail_starters = (
            "for ",
            "in ",
            "of ",
            "from ",
        )

        def _normalize_idx(idx_text: str) -> str:
            normalized = re.sub(r"\s+", "", idx_text.strip("() ")).replace(",", ".")
            return normalized.upper()

        doc = fitz.open(pdf_path)
        pages_lines = []
        for page_num, page in enumerate(doc, start=1):
            page_text = page.get_text("text") or ""
            lines = [re.sub(r"\s+", " ", ln).strip() for ln in page_text.splitlines()]
            pages_lines.append((page_num, lines))

        lof_mode = False
        lof_seen = False
        lof_entries = set()
        lof_first_numbers = set()
        lof_debug_lines = []
        pending_lof_idx = None
        pending_lof_meta = None
        pending_lof_first_num = None
        lof_start_page = None
        lof_max_span_pages = 5

        for page_num, lines in pages_lines:
            for line_num, line in enumerate(lines, start=1):
                if not line:
                    continue

                if lof_heading_pattern.match(line):
                    pending_lof_idx = None
                    pending_lof_meta = None
                    pending_lof_first_num = None
                    lof_mode = True
                    lof_seen = True
                    lof_start_page = page_num
                    if debug:
                        lof_debug_lines.append((page_num, line_num, "lof-heading", line))
                    continue

                if not lof_mode:
                    continue
                if lof_start_page is not None and page_num - lof_start_page > lof_max_span_pages:
                    if pending_lof_idx is not None and debug:
                        p_page, p_line = pending_lof_meta
                        lof_debug_lines.append((p_page, p_line, "lof-standalone-rejected-span", pending_lof_idx))
                    pending_lof_idx = None
                    pending_lof_meta = None
                    pending_lof_first_num = None
                    lof_mode = False
                    if debug:
                        lof_debug_lines.append((page_num, line_num, "lof-end-span", line))
                    continue
                if table_heading_pattern.match(line):
                    if pending_lof_idx is not None and debug:
                        p_page, p_line = pending_lof_meta
                        lof_debug_lines.append((p_page, p_line, "lof-standalone-rejected-table-switch", pending_lof_idx))
                    pending_lof_idx = None
                    pending_lof_meta = None
                    pending_lof_first_num = None
                    lof_mode = False
                    if debug:
                        lof_debug_lines.append((page_num, line_num, "lof-end-table-heading", line))
                    continue

                if generic_heading_pattern.match(line):
                    if pending_lof_idx is not None and debug:
                        p_page, p_line = pending_lof_meta
                        lof_debug_lines.append((p_page, p_line, "lof-standalone-rejected-no-caption", pending_lof_idx))
                    pending_lof_idx = None
                    pending_lof_meta = None
                    pending_lof_first_num = None
                    lof_mode = False
                    if debug:
                        lof_debug_lines.append((page_num, line_num, "lof-end-heading", line))
                    continue

                entry_match = lof_entry_pattern.match(line)
                if entry_match:
                    idx_norm = _normalize_idx(entry_match.group("idx"))
                    lof_entries.add(idx_norm)
                    first_num_match = re.search(r"\d+", idx_norm)
                    if first_num_match:
                        lof_first_numbers.add(int(first_num_match.group()))
                    pending_lof_idx = None
                    pending_lof_meta = None
                    pending_lof_first_num = None
                    if debug:
                        lof_debug_lines.append((page_num, line_num, "lof-entry", line))
                    continue

                standalone_idx_match = lof_standalone_idx_pattern.match(line)
                if standalone_idx_match:
                    idx_norm = _normalize_idx(standalone_idx_match.group("idx"))
                    if pending_lof_idx is not None and debug:
                        p_page, p_line = pending_lof_meta
                        lof_debug_lines.append((p_page, p_line, "lof-standalone-replaced", pending_lof_idx))
                    pending_lof_idx = idx_norm
                    pending_lof_meta = (page_num, line_num)
                    first_num_match = re.search(r"\d+", idx_norm)
                    pending_lof_first_num = int(first_num_match.group()) if first_num_match else None
                    if debug:
                        lof_debug_lines.append((page_num, line_num, "lof-standalone-candidate", line))
                    continue

                idx_caption_match = lof_idx_caption_pattern.match(line)
                if idx_caption_match:
                    idx_norm = _normalize_idx(idx_caption_match.group("idx"))
                    tail = idx_caption_match.group("tail").strip()
                    if caption_letters_pattern.search(tail) and not table_heading_pattern.match(tail):
                        lof_entries.add(idx_norm)
                        first_num_match = re.search(r"\d+", idx_norm)
                        if first_num_match:
                            lof_first_numbers.add(int(first_num_match.group()))
                        if debug:
                            lof_debug_lines.append((page_num, line_num, "lof-leading-idx-entry", line))
                        pending_lof_idx = None
                        pending_lof_meta = None
                        pending_lof_first_num = None
                        continue

                if pending_lof_idx is not None and caption_letters_pattern.search(line):
                    caption_word_count = len(line.split())
                    is_heading_like = bool(lof_heading_pattern.match(line) or generic_heading_pattern.match(line))
                    has_table_switch = bool(table_heading_pattern.match(line))
                    max_seen_first = max(lof_first_numbers) if lof_first_numbers else None
                    is_plausible_first_num = (
                        pending_lof_first_num is not None
                        and pending_lof_first_num <= 60
                        and (
                            max_seen_first is None
                            or pending_lof_first_num <= max_seen_first + 5
                            or pending_lof_first_num in lof_first_numbers
                        )
                    )
                    if is_heading_like or has_table_switch or caption_word_count < 1 or not is_plausible_first_num:
                        if debug:
                            lof_debug_lines.append((page_num, line_num, "lof-standalone-rejected-caption-check", line))
                        continue
                    lof_entries.add(pending_lof_idx)
                    lof_first_numbers.add(pending_lof_first_num)
                    if debug:
                        p_page, p_line = pending_lof_meta
                        lof_debug_lines.append((p_page, p_line, "lof-standalone-accepted", pending_lof_idx))
                        lof_debug_lines.append((page_num, line_num, "lof-caption-for-standalone", line))
                    pending_lof_idx = None
                    pending_lof_meta = None
                    pending_lof_first_num = None
                    continue

                if debug and token_presence_pattern.search(line):
                    lof_debug_lines.append((page_num, line_num, "lof-non-entry", line))

        if pending_lof_idx is not None and debug:
            p_page, p_line = pending_lof_meta
            lof_debug_lines.append((p_page, p_line, "lof-standalone-rejected-eof", pending_lof_idx))

        if lof_seen and lof_entries:
            if debug:
                print(f"[DEBUG num_figures] {pdf_path}")
                print("  mode=fast-track-list-of-figures")
                print(f"  lof_entries={len(lof_entries)}")
                for p, l, reason, text in lof_debug_lines[:120]:
                    print(f"  + p{p}:l{l} [{reason}] {text}")
                if len(lof_debug_lines) > 120:
                    print(f"  ... {len(lof_debug_lines) - 120} more LOF lines omitted")
            doc.close()
            return len(lof_entries)

        unique_keys = set()
        accepted_debug = []
        rejected_debug = []

        for page_num, lines in pages_lines:
            for line_num, line in enumerate(lines, start=1):
                if not line:
                    continue

                lower_line = line.lower()
                has_figure_token = bool(token_presence_pattern.search(line))

                if any(marker in lower_line for marker in ignored_heading_markers):
                    if debug and has_figure_token:
                        rejected_debug.append((page_num, line_num, "heading/list section", line))
                    continue

                if len(line) > 220:
                    if debug and has_figure_token:
                        rejected_debug.append((page_num, line_num, "too long for caption", line))
                    continue

                match_start = caption_start_pattern.match(line)
                if match_start:
                    sep = (match_start.group("sep") or "").strip()
                    tail = (match_start.group("tail") or "").strip().lower()
                    if not sep and any(tail.startswith(starter) for starter in no_sep_reference_tail_starters):
                        if debug:
                            rejected_debug.append((page_num, line_num, "start-line reference fragment (no separator)", line))
                        continue
                    idx_raw = match_start.group("idx")
                    idx_compact = re.sub(r"\s+", "", idx_raw.strip("() "))
                    if (
                        not sep
                        and idx_compact
                        and idx_compact[-1].isalpha()
                        and tail
                    ):
                        split_joined_tail = (idx_compact[-1] + tail).lower()
                        if any(split_joined_tail.startswith(starter.strip()) for starter in no_sep_reference_tail_starters):
                            if debug:
                                rejected_debug.append((page_num, line_num, "start-line reference fragment (split token)", line))
                            continue
                    idx_norm = _normalize_idx(idx_raw)
                    key = (page_num, idx_norm)
                    unique_keys.add(key)
                    if debug:
                        accepted_debug.append((page_num, line_num, "start-of-line caption", line))
                    continue

                match_inline = caption_inline_strong_pattern.search(line)
                if match_inline:
                    prefix = line[: match_inline.start()].lower()
                    if any(cue in prefix for cue in in_text_cues):
                        if debug:
                            rejected_debug.append((page_num, line_num, "in-text reference cue", line))
                        continue

                    if match_inline.start() <= 20 and len(line) <= 160:
                        idx_raw = match_inline.group("idx")
                        idx_norm = _normalize_idx(idx_raw)
                        key = (page_num, idx_norm)
                        unique_keys.add(key)
                        if debug:
                            accepted_debug.append((page_num, line_num, "inline strong fallback", line))
                        continue

                    if debug:
                        rejected_debug.append((page_num, line_num, "weak inline candidate", line))
                    continue

                if debug and has_figure_token:
                    rejected_debug.append((page_num, line_num, "token seen but no valid caption format", line))

        doc.close()

        if debug:
            print(f"[DEBUG num_figures] {pdf_path}")
            print("  mode=fallback-caption-scan")
            if lof_seen:
                print("  note=List-of-Figures heading found but no parseable entries; fallback used")
            print(f"  accepted_candidates={len(accepted_debug)} unique_count={len(unique_keys)}")
            for p, l, reason, text in accepted_debug[:60]:
                print(f"  + p{p}:l{l} [{reason}] {text}")
            if len(accepted_debug) > 60:
                print(f"  ... {len(accepted_debug) - 60} more accepted lines omitted")

            print(f"  rejected_candidates={len(rejected_debug)}")
            for p, l, reason, text in rejected_debug[:80]:
                print(f"  - p{p}:l{l} [{reason}] {text}")
            if len(rejected_debug) > 80:
                print(f"  ... {len(rejected_debug) - 80} more rejected lines omitted")

        return len(unique_keys)
    except Exception as e:
        print(f"[extract_num_figures] Failed for {pdf_path}: {e}")
        return None

### `num_tables` - production

In [9]:
# ==== PRODUCTION (Consolidated): num_tables with LOT fast-track, spaced-dot support, and narrative gate ====
# Dedicated metric implementation: robust List-of-Tables detection + caption fallback.
# Includes: list-likeness gating, body-drift detection, spaced-dot format support, narrative separation filter.


def extract_num_tables(pdf_path: str, debug: bool = False) -> int | None:
    """Estimate number of tables with robust LOT fast-track and caption fallback."""
    try:
        token_pattern = r"(?:table|tab\.?|tabel|t\s*a\s*b(?:\s*l(?:\s*e)?)?\.?)"
        arabic_index_pattern = r"(?:[A-Z]\s*[\.-]\s*)?\d+(?:\s*[\.,-]\s*\d+)*(?:\s*[a-zA-Z])?"
        roman_index_pattern = r"[IVXLCDM]{1,7}"
        index_pattern = rf"\(?\s*(?:{arabic_index_pattern}|{roman_index_pattern})\s*\)?"

        caption_start_pattern = re.compile(
            rf"^\s*(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\s*(?P<sep>[:\-\.,])?\s*(?P<tail>.*)$",
            re.IGNORECASE,
        )
        caption_inline_strong_pattern = re.compile(
            rf"(?<!\w)(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\s*(?P<sep>[:\-])\s*(?P<tail>.+)$",
            re.IGNORECASE,
        )
        token_presence_pattern = re.compile(rf"(?<!\w){token_pattern}(?!\w)", re.IGNORECASE)

        lot_heading_terms = (
            "list of tables",
            "table list",
            "lot",
            "tables",
            "tabeller",
            "tabeloversigt",
            "oversigt over tabeller",
            "fortegnelse over tabeller",
            "liste over tabeller",
            "liste af tabeller",
        )
        numbered_heading_prefix_pattern = rf"(?:{roman_index_pattern}|[A-Z]|\d+(?:\s*[\.-]\s*\d+)*)"
        lot_heading_pattern = re.compile(
            rf"^\s*(?:{numbered_heading_prefix_pattern}\s*[\).:-]?\s+)?(?:{'|'.join(re.escape(term) for term in lot_heading_terms)})\s*:?\s*$",
            re.IGNORECASE,
        )

        lot_entry_pattern = re.compile(
            rf"^\s*(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\b.*$",
            re.IGNORECASE,
        )
        lot_standalone_idx_pattern = re.compile(
            r"^\s*(?P<idx>\d+(?:\s*[\.,-]\s*\d+)*(?:\s*[a-zA-Z])?)\s*$",
            re.IGNORECASE,
        )
        lot_idx_caption_pattern = re.compile(rf"^\s*(?P<idx>{index_pattern})\s+(?P<tail>.+)$", re.IGNORECASE)

        caption_letters_pattern = re.compile(r"[A-Za-zÆØÅæøå]")
        generic_heading_pattern = re.compile(
            r"^\s*(chapter\s+\d+|kapitel\s+\d+|references|litteratur|bibliography|appendix|bilag)\b",
            re.IGNORECASE,
        )
        body_heading_like_pattern = re.compile(
            r"^\s*\d+(?:\s*[\.-]\s*\d+)*(?:\.\d+)*\s+[A-ZÆØÅ][A-Za-zÆØÅæøå]",
            re.IGNORECASE,
        )

        figure_heading_terms = (
            "list of figures",
            "figure list",
            "lof",
            "figures",
            "figurer",
            "figurenliste",
            "figur liste",
            "liste over figurer",
            "figuroversigt",
            "oversigt over figurer",
            "fortegnelse over figurer",
            "liste af figurer",
        )
        figure_heading_pattern = re.compile(
            rf"^\s*(?:{numbered_heading_prefix_pattern}\s*[\).:-]?\s+)?(?:{'|'.join(re.escape(term) for term in figure_heading_terms)})\s*:?\s*$",
            re.IGNORECASE,
        )

        ignored_heading_markers = lot_heading_terms
        in_text_cues = (
            "see",
            "shown in",
            "illustrated in",
            "reported in",
            "as shown in",
            "as seen in",
            "som vist i",
            "se",
            "se også",
            "illustreret i",
            "consider",
            "at",
            "on",
            "again",
            ". ",
            "from",
        )
        no_sep_reference_tail_starters = ("for ", "in ", "of ", "from ")

        # Supports both ".... 10" and ". . . . . 10" (dense and spaced dots)
        dot_leader_with_page_pattern = re.compile(r"(?:\.(?:\s*)){4,}\d{1,4}\s*$")
        trailing_page_pattern = re.compile(r"\b\d{1,4}\s*$")
        lot_line_pattern = re.compile(
            rf"^\s*(?P<idx>{index_pattern})\s+(?P<title>.+?)(?:\.(?:\s*)){4,}(?P<page>\d{{1,4}})\s*$",
            re.IGNORECASE,
        )

        def _normalize_idx(idx_text: str) -> str:
            normalized = re.sub(r"\s+", "", idx_text.strip("() ")).replace(",", ".")
            return normalized.upper()

        def _is_toc_context(lines: list[str], heading_line_num: int) -> bool:
            start = max(0, heading_line_num - 8)
            context = " ".join(lines[start:heading_line_num]).lower()
            toc_markers = ("contents", "preface", "acknowledgements")
            return any(marker in context for marker in toc_markers)

        def _is_list_like_tail(tail: str) -> bool:
            if not tail or not caption_letters_pattern.search(tail):
                return False
            return bool(dot_leader_with_page_pattern.search(tail) or trailing_page_pattern.search(tail))

        doc = fitz.open(pdf_path)
        pages_lines = []
        for page_num, page in enumerate(doc, start=1):
            page_text = page.get_text("text") or ""
            lines = [re.sub(r"\s+", " ", ln).strip() for ln in page_text.splitlines()]
            pages_lines.append((page_num, lines))

        heading_found_page = None
        for scan_page_num, scan_lines in pages_lines:
            for scan_line_num, scan_line in enumerate(scan_lines, start=1):
                if not scan_line:
                    continue
                if lot_heading_pattern.match(scan_line):
                    if not _is_toc_context(scan_lines, scan_line_num - 1):
                        heading_found_page = scan_page_num
                        break
            if heading_found_page is not None:
                break

        mode = False
        seen = False
        entries = set()
        first_numbers = set()
        debug_lines = []
        pending_idx = None
        pending_meta = None
        pending_first_num = None
        start_page = None
        max_span_pages = 12

        # Robustness guards
        probe_max_lines = 60
        probe_lines = 0
        list_evidence_score = 0
        non_list_streak = 0

        scan_start_idx = 0
        if heading_found_page is not None:
            scan_start_idx = next((i for i, (pn, _) in enumerate(pages_lines) if pn == heading_found_page), 0)

        for page_idx, (page_num, lines) in enumerate(pages_lines[scan_start_idx:], start=scan_start_idx):
            for line_num, line in enumerate(lines, start=1):
                if not line:
                    continue

                if lot_heading_pattern.match(line):
                    if _is_toc_context(lines, line_num - 1):
                        if debug:
                            debug_lines.append((page_num, line_num, "num_tables-heading-rejected-toc-context", line))
                        continue
                    pending_idx = None
                    pending_meta = None
                    pending_first_num = None
                    mode = True
                    seen = True
                    start_page = page_num
                    probe_lines = 0
                    list_evidence_score = 0
                    non_list_streak = 0
                    if debug:
                        debug_lines.append((page_num, line_num, "num_tables-heading", line))
                    continue

                if not mode:
                    continue

                if start_page is not None and page_num - start_page > max_span_pages:
                    if pending_idx is not None and debug:
                        p_page, p_line = pending_meta
                        debug_lines.append((p_page, p_line, "num_tables-standalone-rejected-span", pending_idx))
                    pending_idx = None
                    pending_meta = None
                    pending_first_num = None
                    mode = False
                    if debug:
                        debug_lines.append((page_num, line_num, "num_tables-end-span", line))
                    continue

                if figure_heading_pattern.match(line):
                    if pending_idx is not None and debug:
                        p_page, p_line = pending_meta
                        debug_lines.append((p_page, p_line, "num_tables-standalone-rejected-opposite-switch", pending_idx))
                    pending_idx = None
                    pending_meta = None
                    pending_first_num = None
                    mode = False
                    if debug:
                        debug_lines.append((page_num, line_num, "num_tables-end-opposite-heading", line))
                    continue

                if generic_heading_pattern.match(line):
                    if pending_idx is not None and debug:
                        p_page, p_line = pending_meta
                        debug_lines.append((p_page, p_line, "num_tables-standalone-rejected-no-caption", pending_idx))
                    pending_idx = None
                    pending_meta = None
                    pending_first_num = None
                    mode = False
                    if debug:
                        debug_lines.append((page_num, line_num, "num_tables-end-heading", line))
                    continue

                line_is_list_like = False

                lot_line_match = lot_line_pattern.match(line)
                if lot_line_match:
                    idx_norm = _normalize_idx(lot_line_match.group("idx"))
                    entries.add(idx_norm)
                    first_num_match = re.search(r"\d+", idx_norm)
                    if first_num_match:
                        first_numbers.add(int(first_num_match.group()))
                    pending_idx = None
                    pending_meta = None
                    pending_first_num = None
                    list_evidence_score += 2
                    line_is_list_like = True
                    if debug:
                        debug_lines.append((page_num, line_num, "num_tables-lot-line-entry", line))
                    continue

                entry_match = lot_entry_pattern.match(line)
                if entry_match:
                    idx_norm = _normalize_idx(entry_match.group("idx"))
                    entries.add(idx_norm)
                    first_num_match = re.search(r"\d+", idx_norm)
                    if first_num_match:
                        first_numbers.add(int(first_num_match.group()))
                    pending_idx = None
                    pending_meta = None
                    pending_first_num = None
                    list_evidence_score += 2
                    line_is_list_like = True
                    if debug:
                        debug_lines.append((page_num, line_num, "num_tables-entry", line))
                    continue

                idx_match = lot_standalone_idx_pattern.match(line)
                if idx_match:
                    idx_norm = _normalize_idx(idx_match.group("idx"))
                    if pending_idx is not None and debug:
                        p_page, p_line = pending_meta
                        debug_lines.append((p_page, p_line, "num_tables-standalone-replaced", pending_idx))
                    pending_idx = idx_norm
                    pending_meta = (page_num, line_num)
                    first_num_match = re.search(r"\d+", idx_norm)
                    pending_first_num = int(first_num_match.group()) if first_num_match else None
                    if debug:
                        debug_lines.append((page_num, line_num, "num_tables-standalone-candidate", line))
                    continue

                idx_cap_match = lot_idx_caption_pattern.match(line)
                if idx_cap_match:
                    idx_norm = _normalize_idx(idx_cap_match.group("idx"))
                    tail = idx_cap_match.group("tail").strip()
                    tail_lower = tail.lower()

                    if (
                        _is_list_like_tail(tail)
                        and not figure_heading_pattern.match(tail)
                        and not tail_lower.startswith(("references", "bibliography", "contents"))
                    ):
                        entries.add(idx_norm)
                        first_num_match = re.search(r"\d+", idx_norm)
                        if first_num_match:
                            first_numbers.add(int(first_num_match.group()))
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        list_evidence_score += 1
                        line_is_list_like = True
                        if debug:
                            debug_lines.append((page_num, line_num, "num_tables-leading-idx-entry", line))
                        continue

                if pending_idx is not None and caption_letters_pattern.search(line):
                    caption_word_count = len(line.split())
                    is_heading_like = bool(
                        lot_heading_pattern.match(line)
                        or generic_heading_pattern.match(line)
                        or body_heading_like_pattern.match(line)
                    )
                    has_opposite_switch = bool(figure_heading_pattern.match(line))
                    max_seen_first = max(first_numbers) if first_numbers else None
                    is_plausible_first_num = (
                        pending_first_num is not None
                        and pending_first_num <= 60
                        and (
                            max_seen_first is None
                            or pending_first_num <= max_seen_first + 5
                            or pending_first_num in first_numbers
                        )
                    )

                    if (
                        is_heading_like
                        or has_opposite_switch
                        or caption_word_count < 2
                        or not is_plausible_first_num
                        or not _is_list_like_tail(line)
                    ):
                        if debug:
                            debug_lines.append((page_num, line_num, "num_tables-standalone-rejected-caption-check", line))
                        continue

                    accepted_idx = pending_idx
                    accepted_meta = pending_meta
                    entries.add(accepted_idx)
                    first_numbers.add(pending_first_num)
                    pending_idx = None
                    pending_meta = None
                    pending_first_num = None
                    list_evidence_score += 1
                    line_is_list_like = True
                    if debug:
                        p_page, p_line = accepted_meta
                        debug_lines.append((p_page, p_line, "num_tables-standalone-accepted", accepted_idx))
                        debug_lines.append((page_num, line_num, "num_tables-caption-for-standalone", line))
                    continue

                if dot_leader_with_page_pattern.search(line):
                    list_evidence_score += 1
                    line_is_list_like = True

                if line_is_list_like:
                    non_list_streak = 0
                else:
                    non_list_streak += 1

                if probe_lines < probe_max_lines:
                    probe_lines += 1
                    if probe_lines >= 20 and list_evidence_score < 2:
                        if debug:
                            debug_lines.append((page_num, line_num, "num_tables-fasttrack-abort-not-list-like", line))
                        mode = False
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        continue

                # Only enforce drift abort after enough probe context was seen.
                if probe_lines >= 20 and non_list_streak >= 12:
                    if debug:
                        debug_lines.append((page_num, line_num, "num_tables-fasttrack-abort-body-drift", line))
                    mode = False
                    pending_idx = None
                    pending_meta = None
                    pending_first_num = None
                    continue

                if debug and token_presence_pattern.search(line):
                    debug_lines.append((page_num, line_num, "num_tables-non-entry", line))

        if pending_idx is not None and debug:
            p_page, p_line = pending_meta
            debug_lines.append((p_page, p_line, "num_tables-standalone-rejected-eof", pending_idx))

        if seen and len(entries) >= 4:
            if debug:
                print(f"[DEBUG num_tables] {pdf_path}")
                print("  mode=fast-track-list")
                print(f"  entries={len(entries)}")
                print(f"  list_evidence_score={list_evidence_score}, non_list_streak={non_list_streak}")
                for p, l, reason, text in debug_lines[:120]:
                    print(f"  + p{p}:l{l} [{reason}] {text}")
                if len(debug_lines) > 120:
                    print(f"  ... {len(debug_lines) - 120} more lines omitted")
            doc.close()
            return len(entries)

        unique_keys = set()
        accepted_debug = []
        rejected_debug = []

        for page_num, lines in pages_lines:
            for line_num, line in enumerate(lines, start=1):
                if not line:
                    continue

                lower_line = line.lower()
                has_token = bool(token_presence_pattern.search(line))

                if any(marker in lower_line for marker in ignored_heading_markers):
                    if debug and has_token:
                        rejected_debug.append((page_num, line_num, "heading/list section", line))
                    continue

                if len(line) > 220:
                    if debug and has_token:
                        rejected_debug.append((page_num, line_num, "too long for caption", line))
                    continue

                match_start = caption_start_pattern.match(line)
                if match_start:
                    sep = (match_start.group("sep") or "").strip()
                    tail = (match_start.group("tail") or "").strip().lower()
                    if not sep and not dot_leader_with_page_pattern.search(tail):
                        if debug:
                            rejected_debug.append((page_num, line_num, "start-line narrative without separator", line))
                        continue
                    if not sep and any(tail.startswith(starter) for starter in no_sep_reference_tail_starters):
                        if debug:
                            rejected_debug.append((page_num, line_num, "start-line reference fragment (no separator)", line))
                        continue

                    idx_raw = match_start.group("idx")
                    idx_compact = re.sub(r"\s+", "", idx_raw.strip("() "))
                    if not sep and idx_compact and idx_compact[-1].isalpha() and tail:
                        split_joined_tail = (idx_compact[-1] + tail).lower()
                        if any(split_joined_tail.startswith(starter.strip()) for starter in no_sep_reference_tail_starters):
                            if debug:
                                rejected_debug.append((page_num, line_num, "start-line reference fragment (split token)", line))
                            continue

                    idx_norm = _normalize_idx(idx_raw)
                    key = (page_num, idx_norm)
                    unique_keys.add(key)
                    if debug:
                        accepted_debug.append((page_num, line_num, "start-of-line caption", line))
                    continue

                match_inline = caption_inline_strong_pattern.search(line)
                if match_inline:
                    prefix = line[: match_inline.start()].lower()
                    if any(cue in prefix for cue in in_text_cues):
                        if debug:
                            rejected_debug.append((page_num, line_num, "in-text reference cue", line))
                        continue

                    if match_inline.start() <= 20 and len(line) <= 160:
                        idx_raw = match_inline.group("idx")
                        idx_norm = _normalize_idx(idx_raw)
                        key = (page_num, idx_norm)
                        unique_keys.add(key)
                        if debug:
                            accepted_debug.append((page_num, line_num, "inline strong fallback", line))
                        continue

                    if debug:
                        rejected_debug.append((page_num, line_num, "weak inline candidate", line))
                    continue

                if debug and has_token:
                    rejected_debug.append((page_num, line_num, "token seen but no valid caption format", line))

        doc.close()

        if debug:
            print(f"[DEBUG num_tables] {pdf_path}")
            print("  mode=fallback-caption-scan")
            if seen and entries:
                print("  note=List heading found but entries were too uncertain; fallback used")
            print(f"  accepted_candidates={len(accepted_debug)} unique_count={len(unique_keys)}")
            for p, l, reason, text in accepted_debug[:60]:
                print(f"  + p{p}:l{l} [{reason}] {text}")
            if len(accepted_debug) > 60:
                print(f"  ... {len(accepted_debug) - 60} more accepted lines omitted")
            print(f"  rejected_candidates={len(rejected_debug)}")
            for p, l, reason, text in rejected_debug[:80]:
                print(f"  - p{p}:l{l} [{reason}] {text}")
            if len(rejected_debug) > 80:
                print(f"  ... {len(rejected_debug) - 80} more rejected lines omitted")

        return len(unique_keys)
    except Exception as e:
        print(f"[extract_num_tables] Failed for {pdf_path}: {e}")
        return None

### `num_references` - production

In [10]:
def extract_num_references(pdf_path: str, debug: bool = False) -> int | None:
    """Estimate number of references by locating the references section and counting entries."""
    try:
        heading_terms = (
            "references",
            "bibliography",
            "literature",
            "litterature",
            "litteratur",
            "referencer",
            "kilder",
            "litteraturliste",
            "reference list",
        )
        roman_pattern = r"[IVXLCDM]{1,7}"
        numbered_heading_prefix_pattern = rf"(?:{roman_pattern}|[A-Z]|\d+(?:\s*[\.-]\s*\d+)*)"

        refs_heading_pattern = re.compile(
            rf"^\s*(?:{numbered_heading_prefix_pattern}\s*[\).:-]?\s+)?(?:{'|'.join(re.escape(t) for t in heading_terms)})\s*:?\s*(?:[\.-])?\s*(?:\d{1,3})?\s*$",
            re.IGNORECASE,
        )

        stop_heading_pattern = re.compile(
            rf"^\s*(?:{numbered_heading_prefix_pattern}\s*[\).:-]?\s+)?(appendix|appendices|bilag|acknowledg(e)?ments?|about the author|resume|abstract|summary|konklusion|conclusion)\b",
            re.IGNORECASE,
        )

        bracket_num_entry = re.compile(r"^\s*\[(?P<idx>\d{1,4})\]\s+.+")
        bracket_num_standalone = re.compile(r"^\s*\[(?P<idx>\d{1,4})\]\s*$")
        bracket_key_entry = re.compile(r"^\s*\[(?P<key>[A-Za-z][A-Za-z0-9+&\-]{2,20})\]\s*$")
        numeric_entry = re.compile(r"^\s*\(?(?P<idx>\d{1,3})\)?[\.:\)]\s+.+")
        year_only_line = re.compile(r"^\s*(?:19|20)\d{2}[a-z]?\s*[\.,;:]?\s+.+$", re.IGNORECASE)

        author_year_paren_entry = re.compile(
            r"^\s*[A-ZÆØÅ][A-Za-zÆØÅæøå'\-]+(?:,?\s+[A-Z](?:\.|[A-Za-z\-]+))*.*\((?:19|20)\d{2}[a-z]?\)",
            re.IGNORECASE,
        )
        author_year_plain_entry = re.compile(
            r"^\s*[A-ZÆØÅ][A-Za-zÆØÅæøå'\-]+(?:,\s+[A-Z][A-Za-zÆØÅæøå'\-\. ]+){0,8}.*\b(?:19|20)\d{2}[a-z]?\.",
            re.IGNORECASE,
        )
        org_year_entry = re.compile(
            r"^\s*[A-Z0-9][A-Z0-9&/\- ]{1,60}\.\s*(?:19|20)\d{2}[a-z]?\.",
        )

        page_boilerplate = re.compile(
            r"^\s*(page\s+\d+\s+of\s+\d+|master\s+thesis|june\s+\d{4}|\d{5,}/s\d+|s\d{5,}|dtu\b.*)$",
            re.IGNORECASE,
        )

        def is_numbered_entry_start(line: str) -> tuple[bool, int | None, str | None]:
            match_bracket = bracket_num_entry.match(line)
            if match_bracket:
                return True, int(match_bracket.group("idx")), "bracket"

            match_bracket_standalone = bracket_num_standalone.match(line)
            if match_bracket_standalone:
                return True, int(match_bracket_standalone.group("idx")), "bracket-standalone"

            if year_only_line.match(line):
                return False, None, None

            match_numeric = numeric_entry.match(line)
            if match_numeric:
                idx = int(match_numeric.group("idx"))
                if 1900 <= idx <= 2099:
                    return False, None, None
                return True, idx, "numeric"

            return False, None, None

        def is_unnumbered_entry_start(line: str) -> bool:
            if page_boilerplate.match(line):
                return False
            return bool(
                author_year_paren_entry.match(line)
                or author_year_plain_entry.match(line)
                or org_year_entry.match(line)
            )

        def is_toc_context(lines: list[str], heading_line_num: int) -> bool:
            pre_lines = [ln.strip() for ln in lines[:heading_line_num] if ln.strip()]
            context = " ".join(pre_lines).lower()

            toc_markers = (
                "contents",
                "table of contents",
                "indholdsfortegnelse",
                "preface",
                "acknowledgements",
            )
            if any(marker in context for marker in toc_markers):
                return True

            dot_leader_pattern = re.compile(r"(?:\.\s*){4,}\d{1,3}\s*$")
            trailing_page_no_pattern = re.compile(r"\b\d{1,3}\s*$")
            numeric_only_pattern = re.compile(r"^\d{1,3}$")
            toc_tail_markers = ("figurer", "figures", "tabeller", "tables", "bilag", "appendix")
            toc_like_lines = 0
            for ln in pre_lines:
                if dot_leader_pattern.search(ln):
                    toc_like_lines += 1
                    continue
                if trailing_page_no_pattern.search(ln) and re.search(r"[A-Za-zÆØÅæøå]", ln):
                    toc_like_lines += 1

            post_lines = [ln.strip() for ln in lines[heading_line_num + 1 : heading_line_num + 12] if ln.strip()]
            toc_like_post = 0
            post_numeric_only = 0
            post_toc_marker_hits = 0
            for ln in post_lines:
                if dot_leader_pattern.search(ln):
                    toc_like_post += 1
                    continue
                if trailing_page_no_pattern.search(ln) and re.search(r"[A-Za-zÆØÅæøå]", ln):
                    toc_like_post += 1
                if numeric_only_pattern.match(ln):
                    post_numeric_only += 1
                if any(marker in ln.lower() for marker in toc_tail_markers):
                    post_toc_marker_hits += 1

            if toc_like_post >= 3:
                return True
            if post_numeric_only >= 3 and post_toc_marker_hits >= 2:
                return True

            return toc_like_lines >= 6

        def infer_layout_mode(
            bracket_key_starts: int,
            numbered_starts: int,
            unnumbered_starts: int,
        ) -> str:
            if bracket_key_starts >= 1 and numbered_starts == 0:
                return "bracket_key"
            if numbered_starts >= 3 and numbered_starts >= bracket_key_starts + 1:
                return "numbered"
            if unnumbered_starts >= 3 and unnumbered_starts > max(numbered_starts, bracket_key_starts):
                return "unnumbered"
            return "auto"

        doc = fitz.open(pdf_path)
        pages_lines: list[tuple[int, list[str]]] = []
        for page_num, page in enumerate(doc, start=1):
            page_text = page.get_text("text") or ""
            lines = [re.sub(r"\s+", " ", ln).strip() for ln in page_text.splitlines()]
            pages_lines.append((page_num, lines))

        refs_mode = False
        refs_seen = False

        numbered_entries: set[int] = set()
        unnumbered_entries = 0
        in_entry = False

        layout_mode = "auto"
        layout_probe_lines = 0
        layout_probe_max_lines = 80
        probe_bracket_key_starts = 0
        probe_numbered_starts = 0
        probe_unnumbered_starts = 0

        debug_lines: list[tuple[int, int, str, str]] = []

        for page_num, lines in pages_lines:
            for line_num, line in enumerate(lines, start=1):
                if refs_heading_pattern.match(line):
                    if is_toc_context(lines, line_num - 1):
                        if debug:
                            debug_lines.append((page_num, line_num, "refs-heading-rejected-toc", line))
                        continue
                    refs_mode = True
                    refs_seen = True
                    in_entry = False
                    if debug:
                        debug_lines.append((page_num, line_num, "refs-heading", line))
                    continue

                if not refs_mode:
                    continue

                if stop_heading_pattern.match(line):
                    refs_mode = False
                    in_entry = False
                    if debug:
                        debug_lines.append((page_num, line_num, "refs-end-heading", line))
                    continue

                if not line:
                    in_entry = False
                    continue

                if page_boilerplate.match(line):
                    in_entry = False
                    if debug:
                        debug_lines.append((page_num, line_num, "refs-boilerplate", line))
                    continue

                is_bracket_key_start = bool(bracket_key_entry.match(line))
                is_numbered, idx_val, idx_kind = is_numbered_entry_start(line)
                is_unnumbered_start = is_unnumbered_entry_start(line)

                if layout_probe_lines < layout_probe_max_lines:
                    if is_bracket_key_start:
                        probe_bracket_key_starts += 1
                    if is_numbered and idx_val is not None:
                        probe_numbered_starts += 1
                    if is_unnumbered_start:
                        probe_unnumbered_starts += 1
                    layout_probe_lines += 1

                    inferred_mode = infer_layout_mode(
                        probe_bracket_key_starts,
                        probe_numbered_starts,
                        probe_unnumbered_starts,
                    )
                    if inferred_mode != "auto" and inferred_mode != layout_mode:
                        layout_mode = inferred_mode
                        if debug:
                            debug_lines.append(
                                (
                                    page_num,
                                    line_num,
                                    f"refs-layout-lock-{layout_mode}",
                                    line,
                                )
                            )

                if is_bracket_key_start and layout_mode in {"auto", "bracket_key"}:
                    unnumbered_entries += 1
                    in_entry = True
                    if debug:
                        debug_lines.append((page_num, line_num, "refs-entry-bracket-key", line))
                    continue

                if is_numbered and idx_val is not None and layout_mode in {"auto", "numbered"}:
                    numbered_entries.add(idx_val)
                    in_entry = True
                    if debug:
                        debug_lines.append((page_num, line_num, f"refs-entry-numbered-{idx_kind}", line))
                    continue

                if (
                    layout_mode in {"auto", "unnumbered"}
                    and len(numbered_entries) < 3
                    and is_unnumbered_start
                ):
                    unnumbered_entries += 1
                    in_entry = True
                    if debug:
                        debug_lines.append((page_num, line_num, "refs-entry-unnumbered", line))
                    continue

                if in_entry:
                    if debug:
                        debug_lines.append((page_num, line_num, "refs-entry-cont", line))
                    continue

                if debug:
                    debug_lines.append((page_num, line_num, "refs-non-entry", line))

        references_count = max(len(numbered_entries), unnumbered_entries)

        if not refs_seen or references_count == 0:
            tail_start = int(len(pages_lines) * 0.6)
            fallback_numbered: set[int] = set()
            fallback_unnumbered = 0

            for page_num, lines in pages_lines[tail_start:]:
                for line_num, line in enumerate(lines, start=1):
                    if not line or page_boilerplate.match(line):
                        continue
                    is_numbered, idx_val, _ = is_numbered_entry_start(line)
                    if is_numbered and idx_val is not None:
                        fallback_numbered.add(idx_val)
                        if debug:
                            debug_lines.append((page_num, line_num, "refs-fallback-numbered", line))
                        continue
                    if is_unnumbered_entry_start(line):
                        fallback_unnumbered += 1
                        if debug:
                            debug_lines.append((page_num, line_num, "refs-fallback-unnumbered", line))

            fallback_count = max(len(fallback_numbered), fallback_unnumbered)
            if fallback_count >= 5:
                references_count = fallback_count
                if debug:
                    print(f"[DEBUG num_references] {pdf_path}")
                    print("  mode=fallback-tail-scan")
                    print(f"  references_count={references_count}")
                doc.close()
                return references_count

        doc.close()

        if debug:
            print(f"[DEBUG num_references] {pdf_path}")
            print("  mode=references-section-fast-track")
            print(f"  references_count={references_count}")
            print(f"  numbered_unique={len(numbered_entries)}")
            print(f"  unnumbered_count={unnumbered_entries}")
            print(f"  layout_mode={layout_mode}")
            print(
                "  layout_probe_counts="
                f"bracket_key:{probe_bracket_key_starts}, "
                f"numbered:{probe_numbered_starts}, "
                f"unnumbered:{probe_unnumbered_starts}"
            )
            for p, l, reason, text in debug_lines[:120]:
                print(f"  + p{p}:l{l} [{reason}] {text}")
            if len(debug_lines) > 120:
                print(f"  ... {len(debug_lines) - 120} more debug lines omitted")

        return references_count

    except Exception as e:
        print(f"[extract_num_references] Failed for {pdf_path}: {e}")
        return None

### `main` - production

In [11]:
# ==== MAIN EXECUTION (Element 1.1, 1.2 & 1.4) ====
# Production test run for metrics: num_figures, num_tables, and num_references

local_pdf_root = Path("../Data/RAW_test")
pdf_paths = sorted(local_pdf_root.rglob("*.pdf"))

# Debug controls for false-positive inspection
DEBUG_NUM_FIGURES = False  # True
DEBUG_NUM_TABLES = False  # True
DEBUG_NUM_REFERENCES = False  # True
DEBUG_MAX_FILES = 3

### ---- LOAD PDFs ---- ###
if not pdf_paths:
    print(f"No PDFs found under: {local_pdf_root.resolve()}")
    selected_paths = []
else:
    user_choice = (
        input(
            f"Found {len(pdf_paths)} PDFs. Enter number of files to test or 'all': "
        )
        .strip()
        .lower()
    )
    if user_choice == "all":
        selected_paths = pdf_paths
    else:
        try:
            n_files = int(user_choice)
            if n_files <= 0:
                print("Please enter a positive number or 'all'.")
                selected_paths = []
            else:
                selected_paths = pdf_paths[:n_files]
        except ValueError:
            print("Invalid input. Please enter a number or 'all'.")
            selected_paths = []
    print_outputs = input("Print outputs for each file? (y/N): ").strip().lower()

### ---- 1.1, 1.2 & 1.4: Extract metrics ---- ###
records = []
if selected_paths:
    print(f"Testing num_figures, num_tables, and num_references on {len(selected_paths)} PDFs")
    if DEBUG_NUM_FIGURES or DEBUG_NUM_TABLES or DEBUG_NUM_REFERENCES:
        print(f"Debug mode is ON (detailed output for first {DEBUG_MAX_FILES} files)")

    for idx, pdf_path in enumerate(selected_paths):
        try:
            debug_fig = DEBUG_NUM_FIGURES and idx < DEBUG_MAX_FILES
            debug_tab = DEBUG_NUM_TABLES and idx < DEBUG_MAX_FILES
            debug_ref = DEBUG_NUM_REFERENCES and idx < DEBUG_MAX_FILES
            num_figures = extract_num_figures(str(pdf_path), debug=debug_fig)
            num_tables = extract_num_tables(str(pdf_path), debug=debug_tab)
            num_references = extract_num_references(str(pdf_path), debug=debug_ref)
            records.append(
                {
                    "pdf_file": pdf_path.name,
                    "num_figures": num_figures,
                    "num_tables": num_tables,
                    "num_references": num_references,
                }
            )
            if print_outputs == "y":
                print(
                    f"{pdf_path.name}\n  num_figures={num_figures}, num_tables={num_tables}, num_references={num_references}"
                )
        except Exception as e:
            print(f"[MAIN EXECUTION] Failed for {pdf_path}: {e}")
            records.append(
                {
                    "pdf_file": pdf_path.name,
                    "num_figures": None,
                    "num_tables": None,
                    "num_references": None,
                }
            )

# Finalized Element 1 table
element1_df = pd.DataFrame(
    records,
    columns=["pdf_file", "num_figures", "num_tables", "num_references"],
)
print(f"\nBuilt Element 1 results with {len(element1_df)} rows")
display(element1_df)


Testing num_figures, num_tables, and num_references on 12 PDFs

Built Element 1 results with 12 rows


,pdf_file,num_figures,num_tables,num_references
0,5cee68eed9001d2064299318_Deep Reinforcement Le...,20,12,67
1,5cf3aee6d9001d2064299346_Human response to inc...,24,27,14
2,5d04d261d9001d010a45c03d_Comparativ analysis o...,40,14,47
3,5d1c8d6bd9001d146569a4b3_Characterising eye-mo...,46,4,20
4,5d1c8d79d9001d146569a4dc_Deep speech recogniti...,0,0,0
5,5d1c8d7bd9001d146569a4ec_Impact of probiotic t...,13,5,135
6,5d1c8d83d9001d146569a4fd_Storage of heat in th...,67,9,38
7,5d1c8d92d9001d146569a522_Investigating the imp...,38,9,27
8,5d25c7e1d9001d2d454f05ac_Facilitating longitud...,13,3,10
9,5d34488fd9001d016b24698d_Vulnerability assessm...,11,10,75


# DEVELOPMENT


### LEX imports

In [2]:
import spacy
from pathlib import Path
import re
import pandas as pd

### LEX functions

In [3]:
def extract_linguistics_metrics(pdf_path: str, debug: bool = False) -> dict | None:
    """Extract linguistics metrics from thesis PDF, scoped to main content only."""
    try:
        nlp = spacy.load("en_core_web_sm")

        doc = fitz.open(pdf_path)
        num_tot_pages = len(doc)
        min_end_page = max(1, int(num_tot_pages * 0.30))

        end_boundary_exact = {
            "references",
            "bibliography",
            "works cited",
            "list of references",
            "reference list",
            "appendix",
            "appendices",
            "referencer",
            "bibliografi",
            "litteratur",
            "litteraturliste",
            "litteraturfortegnelse",
            "kildeliste",
            "bilag",
            "appendiks",
            "list of figures",
            "list of tables",
        }
        end_boundary_prefix = (
            "references",
            "bibliography",
            "works cited",
            "appendix",
            "appendices",
            "referencer",
            "bibliografi",
            "litteratur",
            "kildeliste",
            "bilag",
            "appendiks",
        )

        def _is_toc_context(lines: list[str], heading_line_num: int) -> bool:
            """Reject heading detections that are likely part of a TOC page/context."""
            pre_lines = [ln.strip() for ln in lines[:heading_line_num] if ln.strip()]
            context = " ".join(pre_lines).lower()

            toc_markers = (
                "contents",
                "table of contents",
                "indholdsfortegnelse",
                "preface",
                "acknowledgements",
            )
            if any(marker in context for marker in toc_markers):
                return True

            dot_leader_pattern = re.compile(r"(?:\.\s*){4,}\d{1,3}\s*$")
            trailing_page_no_pattern = re.compile(r"\b\d{1,3}\s*$")
            numeric_only_pattern = re.compile(r"^\d{1,3}$")
            toc_tail_markers = ("figurer", "figures", "tabeller", "tables", "bilag", "appendix")

            toc_like_lines = 0
            for ln in pre_lines:
                if dot_leader_pattern.search(ln):
                    toc_like_lines += 1
                    continue
                if trailing_page_no_pattern.search(ln) and re.search(r"[A-Za-zÆØÅæøå]", ln):
                    toc_like_lines += 1

            post_lines = [ln.strip() for ln in lines[heading_line_num + 1 : heading_line_num + 12] if ln.strip()]
            toc_like_post = 0
            post_numeric_only = 0
            post_toc_marker_hits = 0
            for ln in post_lines:
                if dot_leader_pattern.search(ln):
                    toc_like_post += 1
                    continue
                if trailing_page_no_pattern.search(ln) and re.search(r"[A-Za-zÆØÅæøå]", ln):
                    toc_like_post += 1
                if numeric_only_pattern.match(ln):
                    post_numeric_only += 1
                if any(marker in ln.lower() for marker in toc_tail_markers):
                    post_toc_marker_hits += 1

            if toc_like_post >= 3:
                return True
            if post_numeric_only >= 3 and post_toc_marker_hits >= 2:
                return True

            return toc_like_lines >= 6

        def _find_main_content_end_page() -> tuple[int, str | None]:
            """Reuse local_metrics boundary logic: return number of content pages and trigger."""
            num_cont_pages = 0
            match_trigger = None

            for page_number, page in enumerate(doc, start=1):
                text = page.get_text("text") or ""
                lines = [line.strip().lower() for line in text.splitlines() if line.strip()]

                matched_line = None
                local_trigger = None

                for line_idx, line in enumerate(lines):
                    tokens = line.split()
                    prefix_token = None
                    core_line = line

                    if tokens:
                        first_token = tokens[0].rstrip(").:-")
                        if first_token.isdigit() or (
                            len(first_token) == 1 and first_token.isalpha()
                        ):
                            prefix_token = first_token
                            core_line = " ".join(tokens[1:]).strip()

                    if core_line and core_line in end_boundary_exact:
                        if prefix_token and prefix_token.isdigit():
                            local_trigger = (
                                f"numeric-prefix exact ('{prefix_token} {core_line}')"
                            )
                        elif prefix_token:
                            local_trigger = (
                                f"letter-prefix exact ('{prefix_token} {core_line}')"
                            )
                        else:
                            local_trigger = f"exact ('{core_line}')"
                    elif core_line and any(core_line.startswith(p) for p in end_boundary_prefix):
                        matched_prefix = next(
                            p for p in end_boundary_prefix if core_line.startswith(p)
                        )
                        if prefix_token and prefix_token.isdigit():
                            local_trigger = (
                                f"numeric-prefix prefix-match ('{prefix_token} {core_line}', prefix='{matched_prefix}')"
                            )
                        elif prefix_token:
                            local_trigger = (
                                f"letter-prefix prefix-match ('{prefix_token} {core_line}', prefix='{matched_prefix}')"
                            )
                        else:
                            local_trigger = (
                                f"prefix-match ('{core_line}', prefix='{matched_prefix}')"
                            )
                    else:
                        local_trigger = None

                    if local_trigger is None:
                        continue

                    words = line.split()
                    has_short_length = len(line) <= 60 and len(words) <= 8
                    ends_with_punctuation = line.endswith(",") or line.endswith(";") or line.endswith(":") or line.endswith(".") or line.endswith(")")

                    core_words = core_line.split()
                    first_core_token = core_words[0] if core_words else ""
                    trailing_words = (
                        core_words[1:] if first_core_token in end_boundary_exact else core_words
                    )
                    lowercase_trailing_count = sum(
                        1 for w in trailing_words if w.isalpha() and w.islower()
                    )
                    has_many_lowercase_trailing = lowercase_trailing_count >= 4

                    if not has_short_length:
                        continue
                    if ends_with_punctuation:
                        continue
                    if has_many_lowercase_trailing:
                        continue

                    # New hardening: ignore TOC-context matches.
                    if _is_toc_context(lines, line_idx):
                        continue

                    matched_line = line
                    break

                if matched_line is not None:
                    if page_number > min_end_page:
                        match_trigger = local_trigger
                        break

                num_cont_pages += 1

            return num_cont_pages, match_trigger

        num_cont_pages, match_trigger = _find_main_content_end_page()

        main_text_parts = []
        for page in doc[:num_cont_pages]:
            main_text_parts.append(page.get_text("text") or "")
        doc.close()

        main_text = "\n".join(main_text_parts)
        main_text = re.sub(r"\s+", " ", main_text).strip()
        if not main_text:
            return None

        nlp_doc = nlp(main_text[:200000])

        sentences = list(nlp_doc.sents)
        words = [token.text for token in nlp_doc if not token.is_punct and not token.is_space]

        if not sentences or not words:
            return None

        total_sentences = len(sentences)
        total_words = len(words)
        unique_words = len(set(w.lower() for w in words))

        avg_sentence_length = total_words / total_sentences
        avg_word_length = sum(len(w) for w in words) / total_words if total_words > 0 else 0
        lexical_diversity = unique_words / total_words if total_words > 0 else 0

        syllable_count = sum(count_syllables(w) for w in words)
        fk_grade = (0.39 * total_words / total_sentences) + (11.8 * syllable_count / total_words) - 15.59
        fk_grade = max(0, min(18, fk_grade))

        metrics = {
            "total_sentences": total_sentences,
            "total_words": total_words,
            "unique_words": unique_words,
            "avg_sentence_length": round(avg_sentence_length, 2),
            "avg_word_length": round(avg_word_length, 2),
            "lexical_diversity": round(lexical_diversity, 3),
            "flesch_kincaid_grade": round(fk_grade, 1),
        }

        if debug:
            print(f"[Linguistics] {Path(pdf_path).name}")
            print(f"  num_tot_pages: {num_tot_pages}")
            print(f"  num_cont_pages: {num_cont_pages}")
            print(f"  content_end_trigger: {match_trigger}")
            for k in (
                "total_sentences",
                "total_words",
                "unique_words",
                "avg_sentence_length",
                "avg_word_length",
                "lexical_diversity",
                "flesch_kincaid_grade",
            ):
                print(f"  {k}: {metrics[k]}")

        return metrics

    except Exception as e:
        print(f"[extract_linguistics_metrics] Failed for {pdf_path}: {e}")
        return None


def count_syllables(word: str) -> int:
    """Approximate syllable count for Flesch-Kincaid calculation."""
    word = word.lower()
    vowels = "aeiou"
    syllable_count = 0
    previous_was_vowel = False

    for char in word:
        is_vowel = char in vowels
        if is_vowel and not previous_was_vowel:
            syllable_count += 1
        previous_was_vowel = is_vowel

    if word.endswith("e"):
        syllable_count -= 1

    return max(1, syllable_count)

### LEX main

In [4]:
# Flexible linguistics run: choose how many files to process

local_pdf_root = Path("../Data/RAW_test")
pdf_paths = sorted(local_pdf_root.rglob("*.pdf"))

# Build candidate file list (prefer validation list when available)
if "validation" in locals() and "pdf_file" in validation.columns:
    candidate_files = [str(x).strip() for x in validation["pdf_file"].dropna().tolist()]
else:
    candidate_files = sorted([p.name for p in local_pdf_root.glob("*.pdf")])

if not candidate_files:
    print(f"No PDF files found in: {local_pdf_root}")
else:
    total_files = len(candidate_files)
    print(f"Found {total_files} PDF file(s) for linguistics in: {local_pdf_root}")

    user_input = input(f"How many files do you want to process? (1-{total_files}): ").strip()

    try:
        num_to_process = int(user_input)
        if num_to_process <= 0:
            raise ValueError
    except ValueError:
        raise SystemExit("Error: Invalid input. Please enter a positive integer. Execution terminated.")

    if num_to_process > total_files:
        choice = input(
            f"Error: Requested {num_to_process} files, but only {total_files} available.\n"
            "Type 'all' to process all files, or 'q' to terminate: "
        ).strip().lower()

        if choice == "all":
            num_to_process = total_files
        elif choice in {"q", "quit"}:
            raise SystemExit("Execution terminated by user. No files were processed.")
        else:
            raise SystemExit("Error: Invalid choice. Execution terminated without processing files.")

    selected_files = candidate_files[:num_to_process]

    linguistics_results = []
    failed_files = []

    for current_file_number, file_name in enumerate(selected_files, start=1):
        pdf_path = local_pdf_root / file_name
        print(f"\n=== Processing file {current_file_number} of {num_to_process} ===")
        print(f"Current file is: {file_name}")

        if not pdf_path.exists():
            print("File not found, skipping.")
            failed_files.append(file_name)
            continue

        result = extract_linguistics_metrics(str(pdf_path), debug=False)
        if result is None:
            failed_files.append(file_name)
            continue

        linguistics_results.append({"pdf_file": file_name, **result})

    linguistics_df = pd.DataFrame(linguistics_results)

    print("\n" + "=" * 100)
    print("LINGUISTICS METRICS SUMMARY (Main Content Only)")
    print("=" * 100)
    display(linguistics_df)

    if not linguistics_df.empty:
        print("\nSTATISTICS ACROSS SELECTED FILES:")
        print(f"  Avg Sentence Length: {linguistics_df['avg_sentence_length'].mean():.2f} ± {linguistics_df['avg_sentence_length'].std():.2f}")
        print(f"  Avg Word Length: {linguistics_df['avg_word_length'].mean():.2f} ± {linguistics_df['avg_word_length'].std():.2f}")
        print(f"  Lexical Diversity: {linguistics_df['lexical_diversity'].mean():.3f} ± {linguistics_df['lexical_diversity'].std():.3f}")
        print(f"  Flesch-Kincaid Grade: {linguistics_df['flesch_kincaid_grade'].mean():.1f} ± {linguistics_df['flesch_kincaid_grade'].std():.1f}")

    if failed_files:
        print(f"\n⚠ Failed files ({len(failed_files)}):")
        for ff in failed_files:
            print(f"  - {ff}")

    print(f"\n✓ Linguistics processing complete! ({len(linguistics_df)} succeeded / {num_to_process} requested)")

Found 50 PDF file(s) for linguistics in: ../Data/RAW_test

=== Processing file 1 of 2 ===
Current file is: 5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf

=== Processing file 2 of 2 ===
Current file is: 5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf

LINGUISTICS METRICS SUMMARY (Main Content Only)


,pdf_file,total_sentences,total_words,unique_words,avg_sentence_length,avg_word_length,lexical_diversity,flesch_kincaid_grade
0,5cee68eed9001d2064299318_Deep Reinforcement Le...,853,16784,2565,19.68,4.7,0.153,10.7
1,5cf3aee6d9001d2064299346_Human response to inc...,779,12908,2538,16.57,4.8,0.197,9.8



STATISTICS ACROSS SELECTED FILES:
  Avg Sentence Length: 18.12 ± 2.20
  Avg Word Length: 4.75 ± 0.07
  Lexical Diversity: 0.175 ± 0.031
  Flesch-Kincaid Grade: 10.2 ± 0.6

✓ Linguistics processing complete! (2 succeeded / 2 requested)


In [5]:
display(linguistics_df.head())

,pdf_file,total_sentences,total_words,unique_words,avg_sentence_length,avg_word_length,lexical_diversity,flesch_kincaid_grade
0,5cee68eed9001d2064299318_Deep Reinforcement Le...,853,16784,2565,19.68,4.7,0.153,10.7
1,5cf3aee6d9001d2064299346_Human response to inc...,779,12908,2538,16.57,4.8,0.197,9.8


# VALIDATION

In [ ]:
validation = pd.DataFrame({
    "pdf_file": [
        "5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf",
        "5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf",
        "5d04d261d9001d010a45c03d_Comparativ analysis of intervention strategies for a high rise building using Fire Brigade Intervention Model indsatsstr.pdf",
        "5d1c8d6bd9001d146569a4b3_Characterising eye-motion patterns in patients with autistic spectrum disorders using machine learning (translated Karak.pdf",
        "5d1c8d79d9001d146569a4dc_Deep speech recognition in Danish (translated Dyb talegenkendelse pa dansk).pdf",
        "5d1c8d7bd9001d146569a4ec_Impact of probiotic tropodithietic acid (TDA) producing Phaeobacter piscinae on microbial community members related to a.pdf",
        "5d1c8d83d9001d146569a4fd_Storage of heat in the pipeline of existing district heating grid with heat pump supply (translated Lagring af varme i r.pdf",
        "5d1c8d92d9001d146569a522_Investigating the impact of blade desing parameters on blade stability using CFD (translated Undersøgelse af virkningen .pdf",
        "5d25c7e1d9001d2d454f05ac_Facilitating longitudinal interaction in a learning environment, using microgoals.pdf",
        "5d34488fd9001d016b24698d_Vulnerability assessment of European fisheries to climate change (translated Analyse af Europas fiskeris sarbarhed over .pdf",
        "5d3d8341d9001d32f558c139_Application-Specific Hardware to Accelerate Neural Network Training (translated Hardwareaccelerator til træning af neura.pdf",
        "5d3d836cd9001d32f558c193_Optimal Scheduling of Railway Infrastructure Maintenance to Minimize Disruption (translated Optimal Planlæggning af Jern.pdf"
                 ],
    "num_figures": [
        20,24,40,46,None,12,62,38,13,11,28,28
    ],
    "num_tables": [12,26,14,4,None,5,9,9,3,12,13,14],
    "num_equations": [0,0,0,0,0,0,0,0,0,0,0,0], # NOT DATA ATM.
    "num_references": [67,23,47,15,None,124,38,27,11,78,25,56],
    "num_citations": [0,0,0,0,0,0,0,0,0,0,0,0] # NOT DATA ATM.
})
#display(validation[["pdf_file", "num_figures"]])

## - VALIDATION FOR ``num_figures``

In [ ]:
# ==== VALIDATION EVALUATION: num_figures vs true counts ====
from pathlib import Path
import pandas as pd

validation_eval = validation.copy()
validation_eval = validation_eval[validation_eval["num_figures"].notna()].copy()

raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")

predictions = []
missing_files = []
for _, row in validation_eval.iterrows():
    file_name = str(row["pdf_file"]).strip()
    pdf_path = raw_root / file_name
    if not pdf_path.exists():
        missing_files.append(file_name)
        predictions.append(None)
        continue
    pred = extract_num_figures(str(pdf_path), debug=False)
    predictions.append(pred)

validation_eval["pred_num_figures"] = predictions
validation_eval["abs_error"] = (validation_eval["pred_num_figures"] - validation_eval["num_figures"]).abs()
validation_eval["signed_error"] = validation_eval["pred_num_figures"] - validation_eval["num_figures"]

# Tolerance metrics for quick quality checks
validation_eval["within_1"] = validation_eval["abs_error"] <= 1
validation_eval["within_2"] = validation_eval["abs_error"] <= 2
validation_eval["within_10pct"] = validation_eval["abs_error"] <= (0.1 * validation_eval["num_figures"])

# Pass if at least one check is True.
# within_1 / within_2 being True always passes regardless of within_10pct.
validation_eval["pass_validation"] = (
    validation_eval["within_1"]
    | validation_eval["within_2"]
    | validation_eval["within_10pct"]
)

n_total = len(validation_eval)
n_scored = int(validation_eval["pred_num_figures"].notna().sum())
mae = validation_eval["abs_error"].dropna().mean() if n_scored else None
mape = (
    (validation_eval["abs_error"] / validation_eval["num_figures"]).replace([float("inf")], pd.NA).dropna().mean() * 100
    if n_scored else None
)
hit_within_1 = int(validation_eval["within_1"].sum()) if n_scored else 0
hit_within_2 = int(validation_eval["within_2"].sum()) if n_scored else 0
hit_within_10pct = int(validation_eval["within_10pct"].sum()) if n_scored else 0
n_pass = int(validation_eval["pass_validation"].sum()) if n_scored else 0

print("Validation summary (num_figures)")
print(f"- files in validation (non-null true counts): {n_total}")
print(f"- files scored: {n_scored}")
print(f"- MAE: {mae:.2f}" if mae is not None else "- MAE: N/A")
print(f"- MAPE: {mape:.2f}%" if mape is not None else "- MAPE: N/A")
print(f"- within ±1: {hit_within_1}/{n_scored}")
print(f"- within ±2: {hit_within_2}/{n_scored}")
print(f"- within ±10% of true value: {hit_within_10pct}/{n_scored}")
print(f"- pass_validation (>=1/3 checks): {n_pass}/{n_scored}")

if missing_files:
    print("\nMissing files (not scored):")
    for f in missing_files:
        print(f"- {f}")

display(
    validation_eval[[
        "pdf_file",
        "num_figures",
        "pred_num_figures",
        "signed_error",
        "abs_error",
        "within_1",
        "within_2",
        "within_10pct",
        "pass_validation",
    ]].sort_values("abs_error", ascending=False)
)

## - VALIDATION FOR ``num_tables``

In [ ]:
# ==== QUICK VALIDATION (1.2: num_tables on first 12 labeled files) ====
raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")
validation_tables = validation.copy()
validation_tables = validation_tables[validation_tables["num_tables"].notna()].copy()
pred_tables = []
missing_tables = []
for _, row in validation_tables.iterrows():
    file_name = str(row["pdf_file"]).strip()
    pdf_path = raw_root / file_name
    if not pdf_path.exists():
        missing_tables.append(file_name)
        pred_tables.append(None)
        continue
    pred_tables.append(extract_num_tables(str(pdf_path), debug=False))
validation_tables["pred_num_tables"] = pred_tables
validation_tables["abs_error"] = (validation_tables["pred_num_tables"] - validation_tables["num_tables"]).abs()
validation_tables["within_1"] = validation_tables["abs_error"] <= 1
validation_tables["within_2"] = validation_tables["abs_error"] <= 2
validation_tables["within_10pct"] = validation_tables["abs_error"] <= (0.10 * validation_tables["num_tables"])

# Pass if at least one check is True.
# within_1 / within_2 being True always passes regardless of within_10pct.
validation_tables["pass_validation"] = (
    validation_tables["within_1"]
    | validation_tables["within_2"]
    | validation_tables["within_10pct"]
)

n_scored = int(validation_tables["pred_num_tables"].notna().sum())
hit_10pct = int(validation_tables["within_10pct"].sum()) if n_scored else 0
mae = validation_tables["abs_error"].dropna().mean() if n_scored else None
n_pass = int(validation_tables["pass_validation"].sum()) if n_scored else 0
print("Quick validation summary (num_tables)")
print(f"- files scored: {n_scored}")
print(f"- MAE: {mae:.2f}" if mae is not None else "- MAE: N/A")
print(f"- within ±1: {validation_tables['within_1'].sum()}/{n_scored}")
print(f"- within ±2: {validation_tables['within_2'].sum()}/{n_scored}")
print(f"- within ±10%: {hit_10pct}/{n_scored}")
print(f"- pass_validation (>=1/3 checks): {n_pass}/{n_scored}")
if missing_tables:
    print("\nMissing files (not scored):")
    for f in missing_tables:
        print(f"- {f}")
display(
    validation_tables[[
        "pdf_file",
        "num_tables",
        "pred_num_tables",
        "abs_error",
        "within_1",
        "within_2",
        "within_10pct",
        "pass_validation",
    ]].sort_values("abs_error", ascending=False)
)

## - VALIDATION FOR ``num_references``

In [ ]:
# ==== QUICK VALIDATION (1.4: num_references on first 12 labeled files) ====
raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")
validation_tables = validation.copy()
validation_tables = validation_tables[validation_tables["num_references"].notna()].copy()
pred_tables = []
missing_tables = []
for _, row in validation_tables.iterrows():
    file_name = str(row["pdf_file"]).strip()
    pdf_path = raw_root / file_name
    if not pdf_path.exists():
        missing_tables.append(file_name)
        pred_tables.append(None)
        continue
    pred_tables.append(extract_num_references(str(pdf_path), debug=False))
validation_tables["pred_num_references"] = pred_tables
validation_tables["abs_error"] = (validation_tables["pred_num_references"] - validation_tables["num_references"]).abs()
validation_tables["within_1"] = validation_tables["abs_error"] <= 1
validation_tables["within_2"] = validation_tables["abs_error"] <= 2
validation_tables["within_10pct"] = validation_tables["abs_error"] <= (0.10 * validation_tables["num_references"])

# Pass if at least one check is True.
# within_1 / within_2 being True always passes regardless of within_10pct.
validation_tables["pass_validation"] = (
    validation_tables["within_1"]
    | validation_tables["within_2"]
    | validation_tables["within_10pct"]
)

n_scored = int(validation_tables["pred_num_references"].notna().sum())
hit_10pct = int(validation_tables["within_10pct"].sum()) if n_scored else 0
mae = validation_tables["abs_error"].dropna().mean() if n_scored else None
n_pass = int(validation_tables["pass_validation"].sum()) if n_scored else 0
print("Quick validation summary (num_references)")
print(f"- files scored: {n_scored}")
print(f"- MAE: {mae:.2f}" if mae is not None else "- MAE: N/A")
print(f"- within ±1: {validation_tables['within_1'].sum()}/{n_scored}")
print(f"- within ±2: {validation_tables['within_2'].sum()}/{n_scored}")
print(f"- within ±10%: {hit_10pct}/{n_scored}")
print(f"- pass_validation (>=1/3 checks): {n_pass}/{n_scored}")
if missing_tables:
    print("\nMissing files (not scored):")
    for f in missing_tables:
        print(f"- {f}")
display(
    validation_tables[[
        "pdf_file",
        "num_references",
        "pred_num_references",
        "abs_error",
        "within_1",
        "within_2",
        "within_10pct",
        "pass_validation",
    ]].sort_values("abs_error", ascending=False)
)

# HELPFULL CODE CELLS

## - DEBUG n PRINT: to get the outputs for rootcause analysis - choose metric to analyse

In [ ]:
# ==== ROOTCAUSE CONFIG (Cells 22-24) ====
from pathlib import Path
import re
import fitz
import json

raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")

hardcoded_debug_files = [
    "5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf",
    "5d1c8d6bd9001d146569a4b3_Characterising eye-motion patterns in patients with autistic spectrum disorders using machine learning (translated Karak.pdf",
    "5d3d8341d9001d32f558c139_Application-Specific Hardware to Accelerate Neural Network Training (translated Hardwareaccelerator til træning af neura.pdf",
]

valid_metrics = ("num_figures", "num_tables", "num_references")
metric_choice = input(
    "Metric for rootcause analysis (num_figures/num_tables/num_references): "
).strip().lower()
if metric_choice not in valid_metrics:
    print("Invalid choice. Defaulting to num_tables.")
    metric_choice = "num_tables"

file_source = input(
    "File source [validation/hardcoded] (default validation): "
).strip().lower()
if file_source not in {"", "validation", "hardcoded"}:
    print("Invalid source. Defaulting to validation.")
    file_source = "validation"
if file_source == "":
    file_source = "validation"

def _parse_row_selection(selection_text: str, max_row: int) -> list[int]:
    selection_text = selection_text.strip().lower()
    if selection_text in {"", "all"}:
        return list(range(1, max_row + 1))

    selected = []
    for part in selection_text.split(","):
        token = part.strip()
        if not token:
            continue
        if "-" in token:
            bounds = token.split("-", 1)
            if len(bounds) != 2 or not bounds[0].isdigit() or not bounds[1].isdigit():
                continue
            start = int(bounds[0])
            end = int(bounds[1])
            if start > end:
                start, end = end, start
            for row_num in range(start, end + 1):
                if 1 <= row_num <= max_row:
                    selected.append(row_num)
        else:
            if token.isdigit():
                row_num = int(token)
                if 1 <= row_num <= max_row:
                    selected.append(row_num)

    deduped = []
    seen = set()
    for row_num in selected:
        if row_num not in seen:
            deduped.append(row_num)
            seen.add(row_num)
    return deduped

debug_files = []
if file_source == "hardcoded":
    debug_files = hardcoded_debug_files[:]
else:
    if "validation" not in globals():
        raise RuntimeError(
            "validation DataFrame is not in memory. Run the validation table cell first or choose hardcoded source."
        )

    if "pdf_file" not in validation.columns:
        raise RuntimeError("validation DataFrame must include a 'pdf_file' column.")

    if metric_choice in validation.columns:
        validation_candidates = validation[validation[metric_choice].notna()].copy()
        if validation_candidates.empty:
            print(
                f"No non-null '{metric_choice}' labels in validation table. Falling back to all validation rows."
            )
            validation_candidates = validation.copy()
    else:
        print(f"'{metric_choice}' not found in validation columns. Using all validation rows.")
        validation_candidates = validation.copy()

    validation_candidates = validation_candidates.reset_index(drop=True)
    if validation_candidates.empty:
        raise RuntimeError("Validation table has no rows available for selection.")

    preview_count_text = input(
        f"How many validation rows to preview? (default {min(12, len(validation_candidates))}): "
).strip()
    try:
        preview_count = int(preview_count_text) if preview_count_text else min(12, len(validation_candidates))
        if preview_count < 1:
            preview_count = min(12, len(validation_candidates))
    except ValueError:
        preview_count = min(12, len(validation_candidates))

    preview_count = min(preview_count, len(validation_candidates))
    print("\nValidation rows available (1-based selection index):")
    for idx in range(preview_count):
        file_name = str(validation_candidates.loc[idx, "pdf_file"])
        if metric_choice in validation_candidates.columns:
            true_val = validation_candidates.loc[idx, metric_choice]
            print(f"{idx + 1:>2}. {file_name} | true_{metric_choice}={true_val}")
        else:
            print(f"{idx + 1:>2}. {file_name}")
    if preview_count < len(validation_candidates):
        print(f"... plus {len(validation_candidates) - preview_count} more rows")

    row_selection = input(
        "Select rows (e.g. 1,3,5,9 or 1-5 or 1,3-5,9 or all): "
).strip()
    selected_rows = _parse_row_selection(row_selection, len(validation_candidates))
    if not selected_rows:
        print("No valid row selection parsed. Defaulting to all rows.")
        selected_rows = list(range(1, len(validation_candidates) + 1))

    debug_files = [
        str(validation_candidates.loc[row_num - 1, "pdf_file"]).strip()
        for row_num in selected_rows
    ]

print_debug_output = input("Print extractor debug output? (Y/n): ").strip().lower() != "n"
print_page_text = input("Print page text around detected heading? (Y/n): ").strip().lower() != "n"
pages_after_heading = input("How many pages to print from heading page? (default 2): ").strip()
try:
    pages_after_heading = int(pages_after_heading) if pages_after_heading else 2
    if pages_after_heading < 1:
        pages_after_heading = 2
except ValueError:
    pages_after_heading = 2

def _ensure_extractor_loaded(metric_name: str):
    target_func_name = f"extract_{metric_name}"
    if target_func_name in globals() and callable(globals()[target_func_name]):
        return globals()[target_func_name]

    nb_candidates = [
        Path("thesis_stats_extractor.ipynb"),
        Path("exstraction_more_from_pdf/thesis_stats_extractor.ipynb"),
    ]
    nb_path = next((p for p in nb_candidates if p.exists()), None)
    if nb_path is None:
        raise RuntimeError(
            f"Could not find thesis_stats_extractor.ipynb to load {target_func_name}."
        )

    nb = json.loads(nb_path.read_text())
    source_to_exec = None
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = "\n".join(cell.get("source", []))
        if f"def {target_func_name}(" in src:
            source_to_exec = src
            break

    if source_to_exec is None:
        raise RuntimeError(f"Could not locate {target_func_name} in notebook source.")

    exec(source_to_exec, globals())
    if target_func_name not in globals() or not callable(globals()[target_func_name]):
        raise RuntimeError(f"Failed to load {target_func_name} into kernel.")
    return globals()[target_func_name]

selected_extractor = _ensure_extractor_loaded(metric_choice)

heading_terms_map = {
    "num_figures": (
        "list of figures",
        "figure list",
        "lof",
        "figures",
        "figurer",
        "figurenliste",
        "figur liste",
        "liste over figurer",
        "figuroversigt",
        "oversigt over figurer",
        "fortegnelse over figurer",
        "liste af figurer",
    ),
    "num_tables": (
        "list of tables",
        "table list",
        "lot",
        "tables",
        "tabeller",
        "tabeloversigt",
        "oversigt over tabeller",
        "fortegnelse over tabeller",
        "liste over tabeller",
        "liste af tabeller",
    ),
    "num_references": (
        "references",
        "bibliography",
        "literature",
        "litterature",
        "litteratur",
        "referencer",
        "kilder",
        "litteraturliste",
        "reference list",
    ),
}

heading_terms = heading_terms_map[metric_choice]
roman_pattern = r"[IVXLCDM]{1,7}"
numbered_prefix = rf"(?:{roman_pattern}|[A-Z]|\d+(?:\s*[\.-]\s*\d+)*)"
metric_heading_pattern = re.compile(
    rf"^\s*(?:{numbered_prefix}\s*[\).:-]?\s+)?(?:{'|'.join(re.escape(t) for t in heading_terms)})\s*:?\s*(?:[\.-])?\s*(?:\d{1,3})?\s*$",
    re.IGNORECASE,
)

print("\nRootcause config ready:")
print(f"- metric_choice={metric_choice}")
print(f"- file_source={file_source}")
print(f"- files_selected={len(debug_files)}")
for idx, file_name in enumerate(debug_files, start=1):
    print(f"  {idx}. {file_name}")
print(f"- print_debug_output={print_debug_output}")
print(f"- print_page_text={print_page_text}, pages_after_heading={pages_after_heading}")

In [ ]:
# ==== ROOTCAUSE DEBUG RUN (uses config from Cell 22) ====
import io
import contextlib

if "selected_extractor" not in globals() or "metric_choice" not in globals():
    raise RuntimeError("Run Cell 22 first to set metric/files/debug config.")

print("\n" + "=" * 120)
print(f"DEBUG RUN for {metric_choice}")
print("=" * 120)

for file_name in debug_files:
    pdf_path = raw_root / file_name
    print("\n" + "-" * 120)
    print(f"FILE: {file_name}")

    if not pdf_path.exists():
        print("Status: MISSING FILE")
        continue

    if print_debug_output:
        capture = io.StringIO()
        with contextlib.redirect_stdout(capture):
            pred_value = selected_extractor(str(pdf_path), debug=True)
        debug_lines = capture.getvalue().splitlines()

        print(f"pred_{metric_choice}={pred_value}")
        print(f"debug_lines={len(debug_lines)}")
        if debug_lines:
            print("\n[Debug Output Preview]")
            for ln in debug_lines[:200]:
                print(ln)
            if len(debug_lines) > 200:
                print(f"... {len(debug_lines) - 200} more debug lines omitted")
    else:
        pred_value = selected_extractor(str(pdf_path), debug=False)
        print(f"pred_{metric_choice}={pred_value}")
        print("Debug output skipped (print_debug_output=False)")

In [ ]:
# ==== ROOTCAUSE PAGE PRINT (uses config from Cell 22) ====
if "metric_heading_pattern" not in globals() or "metric_choice" not in globals():
    raise RuntimeError("Run Cell 22 first to set metric/files/page-print config.")

if not print_page_text:
    print("Page text printing is disabled (print_page_text=False).")
else:
    for file_name in debug_files:
        pdf_path = raw_root / file_name
        print("\n" + "=" * 140)
        print(f"FILE: {file_name}")

        if not pdf_path.exists():
            print("Status: MISSING FILE")
            continue

        doc = fitz.open(str(pdf_path))
        heading_page = None

        def _is_metric_heading_toc(lines: list[str], heading_line_num: int) -> bool:
            pre_lines = [ln.strip() for ln in lines[:heading_line_num] if ln.strip()]
            context = " ".join(pre_lines).lower()
            toc_markers = (
                "contents",
                "table of contents",
                "indholdsfortegnelse",
                "preface",
                "acknowledgements",
            )
            if any(marker in context for marker in toc_markers):
                return True

            dot_leader_pattern = re.compile(r"(?:\.\s*){4,}\d{1,3}\s*$")
            trailing_page_no_pattern = re.compile(r"\b\d{1,3}\s*$")
            numeric_only_pattern = re.compile(r"^\d{1,3}$")
            toc_tail_markers = ("figurer", "figures", "tabeller", "tables", "bilag", "appendix")
            toc_like_lines = 0
            for ln in pre_lines:
                if dot_leader_pattern.search(ln):
                    toc_like_lines += 1
                    continue
                if trailing_page_no_pattern.search(ln) and re.search(r"[A-Za-zÆØÅæøå]", ln):
                    toc_like_lines += 1

            post_lines = [ln.strip() for ln in lines[heading_line_num + 1 : heading_line_num + 12] if ln.strip()]
            toc_like_post = 0
            post_numeric_only = 0
            post_toc_marker_hits = 0
            for ln in post_lines:
                if dot_leader_pattern.search(ln):
                    toc_like_post += 1
                    continue
                if trailing_page_no_pattern.search(ln) and re.search(r"[A-Za-zÆØÅæøå]", ln):
                    toc_like_post += 1
                if numeric_only_pattern.match(ln):
                    post_numeric_only += 1
                if any(marker in ln.lower() for marker in toc_tail_markers):
                    post_toc_marker_hits += 1

            if toc_like_post >= 3:
                return True
            if post_numeric_only >= 3 and post_toc_marker_hits >= 2:
                return True
            return toc_like_lines >= 6

        for page_idx in range(len(doc)):
            text = doc[page_idx].get_text("text") or ""
            lines = [re.sub(r"\s+", " ", ln).strip() for ln in text.splitlines()]
            for line_num, ln in enumerate(lines, start=1):
                if not ln:
                    continue
                if metric_heading_pattern.match(ln) and not _is_metric_heading_toc(lines, line_num - 1):
                    heading_page = page_idx
                    break
            if heading_page is not None:
                break

        if heading_page is None:
            inferred_page = None
            if metric_choice == "num_references":
                bracket_num_line = re.compile(r"^\s*\[\d{1,4}\]\s*(?:.+)?$")
                numeric_ref_line = re.compile(r"^\s*\(?(?:\d{1,3})\)?[\.:\)]\s+.+$")
                tail_start = int(len(doc) * 0.6)
                for page_idx in range(tail_start, len(doc)):
                    text = doc[page_idx].get_text("text") or ""
                    lines = [re.sub(r"\s+", " ", ln).strip() for ln in text.splitlines()]
                    ref_like_starts = 0
                    for ln in lines:
                        if bracket_num_line.match(ln) or numeric_ref_line.match(ln):
                            ref_like_starts += 1
                    if ref_like_starts >= 4:
                        inferred_page = page_idx
                        break

            print(f"No heading found for metric: {metric_choice}")
            if inferred_page is not None:
                print(f"Using inferred references-like page (1-based): {inferred_page + 1}")
                last_idx = min(len(doc), inferred_page + pages_after_heading)
                page_indexes = list(range(inferred_page, last_idx))
            else:
                print("Printing the last page as fallback for quick inspection.")
                page_indexes = [max(0, len(doc) - 1)]
        else:
            print(f"Detected heading page (1-based): {heading_page + 1}")
            last_idx = min(len(doc), heading_page + pages_after_heading)
            page_indexes = list(range(heading_page, last_idx))

        for page_idx in page_indexes:
            print("\n" + "-" * 60)
            print(f"PAGE {page_idx + 1}")
            print("-" * 60)
            page_text = doc[page_idx].get_text("text") or ""
            print(page_text[:7000])

        doc.close()

## - PLOT DISTRIBUTION - choose what metric to plot

In [ ]:
# user to choose what metric to analyze
metric_choice = input("Enter metric to analyze (num_figures/num_tables/num_references): ").strip().lower()
if metric_choice == "num_figures":
    df_to_analyze = num_figures_df
    title = "Figures"
elif metric_choice == "num_tables":
    df_to_analyze = num_tables_df
    title = "Tables"
elif metric_choice == "num_references":
    df_to_analyze = num_references_df
    title = "References"
else:
    print("Invalid choice.")

# summarary statistics on num_figures
print(f"\nSummary statistics for {metric_choice}:")
print(df_to_analyze[metric_choice].describe())
# barplot of num_figures distribution
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
df_to_analyze[metric_choice].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title(f'Distribution of Number of {title} Extracted')
plt.xlabel(f'Number of {title}')
plt.ylabel('Count of PDFs')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# display 10 larges values in for df:
print("\nTop 10 PDFs with highest {metric_choice}:")
display(df_to_analyze.sort_values(by=metric_choice, ascending=False).head(10))

## - QUICK PDF PAGE PRINTS

In [ ]:
# ==== QUICK TEST CELL ====
### ---- PRINT X PAGES OF A SINGLE PDF ---- ###
import pdfplumber
pdf_input = "Data/RAW_test/5d1c8d7bd9001d146569a4ec_Impact of probiotic tropodithietic acid (TDA) producing Phaeobacter piscinae on microbial community members related to a.pdf"
# Use existing variables from the notebook
pdf_file = Path(pdf_input) if "pdf_input" in globals() else Path(pdf_path) / pdf_file_name
# Optional fallback if relative path differs by working directory
if not pdf_file.exists():
    alt = Path("../") / pdf_file
    if alt.exists():
        pdf_file = alt
x_pages = int(input("How many pages to print? ").strip())
with pdfplumber.open(str(pdf_file)) as pdf:
    total_pages = len(pdf.pages)
    n = min(x_pages, total_pages)
    print(f"Printing {n}/{total_pages} page(s) from: {pdf_file.name}\n")
    for i in range(n):
        text = pdf.pages[i].extract_text() or "[No extractable text]"
        print(f"{'=' * 20} PAGE {i + 1} {'=' * 20}")
        print(text)
        print()

In [ ]:
# ==== QUICK TEST CELL ====
### ---- PRINT PAGE X OF A SINGLE PDF ---- ###
import pdfplumber
pdf_input = "Data/RAW_test/5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf"
# Keep current pdf_file logic if needed
pdf_file = Path(pdf_input) if "pdf_input" in globals() else Path(pdf_path) / pdf_file_name
if not pdf_file.exists():
    alt = Path("../") / pdf_file
    if alt.exists():
        pdf_file = alt
page_to_print = int(input("Which page to print? (1-based): ").strip())
with pdfplumber.open(str(pdf_file)) as pdf:
    total_pages = len(pdf.pages)
    if not (1 <= page_to_print <= total_pages):
        print(f"Invalid page number. Enter a value between 1 and {total_pages}.")
    else:
        i = page_to_print - 1  # convert to 0-based index
        text = pdf.pages[i].extract_text() or "[No extractable text]"
        print(f"Printing page {page_to_print}/{total_pages} from: {pdf_file.name}\n")
        print(f"{'=' * 20} PAGE {page_to_print} {'=' * 20}")
        print(text)

# ARCHIVES

## - DISPLAY FILES THAT FAILED PASS VALUDATION

In [ ]:
# Identify failing files for num_figures
print("\n" + "=" * 120)
print("FAILING FILES FOR NUM_FIGURES (pass_validation=False)")
print("=" * 120)

failing_files = validation_eval[validation_eval["pass_validation"] == False].copy()
failing_files = failing_files.sort_values("abs_error", ascending=False)

print(f"\nTotal failing: {len(failing_files)}/11\n")
for idx, row in failing_files.iterrows():
    print(f"❌ {row['pdf_file']}")
    print(f"   True: {int(row['num_figures'])}, Predicted: {int(row['pred_num_figures'])}, Error: {int(row['abs_error'])} (±{row['abs_error']/row['num_figures']*100:.1f}%)")
    print(f"   within_1: {row['within_1']}, within_2: {row['within_2']}, within_10pct: {row['within_10pct']}")
    print()


## - TARGETED (specefic file) VALIDATION CHECK

In [ ]:
# ==== ARCHIVE: TARGETED VALIDATION CHECK (num_figures) ====
# HOW TO USE
# 1) Set TARGET_FILE to a filename present in validation['pdf_file'].
# 2) Ensure Cell 8 (validation table) and Cell 5 (extract_num_figures) are already run.
# 3) Run this cell.
# ---------------------------------
# HOW TO READ
# - true_num_figures: expected value from validation table.
# - pred_num_figures: model prediction from extract_num_figures.
# - abs_error: absolute difference.
# - pct_error: absolute percent error relative to true value.
# - within_10pct: True means pass under the current tolerance rule.
from pathlib import Path
# Change this filename when doing a one-file diagnostic.
TARGET_FILE = "5d1c8d6bd9001d146569a4b3_Characterising eye-motion patterns in patients with autistic spectrum disorders using machine learning (translated Karak.pdf"
true_num_figures = float(
    validation.loc[validation["pdf_file"] == TARGET_FILE, "num_figures"].iloc[0]
)
raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")
target_pdf = raw_root / TARGET_FILE
pred_num_figures = extract_num_figures(str(target_pdf), debug=False)
abs_error = abs(pred_num_figures - true_num_figures)
pct_error = (abs_error / true_num_figures) * 100
print("target:", target_pdf.name)
print("true_num_figures:", true_num_figures)
print("pred_num_figures:", pred_num_figures)
print("abs_error:", abs_error)
print("pct_error:", round(pct_error, 2), "%")
print("within_10pct:", pct_error <= 10)

## - FIGURES DIAGNOSTICS: current production vs git-base

In [ ]:
# ==== FIGURES DIAGNOSTICS: current production vs git-base ====
import json
import re
import subprocess
from pathlib import Path

import fitz
import pandas as pd

current_nb = json.loads(Path("thesis_stats_extractor.ipynb").read_text())
base_nb = json.loads(
    subprocess.check_output(
        ["git", "show", "HEAD:exstraction_more_from_pdf/thesis_stats_extractor.ipynb"],
        text=True,
    )
)

cur_src = None
base_src = None
for cell in current_nb.get("cells", []):
    if cell.get("cell_type") != "code":
        continue
    src = "\n".join(cell.get("source", []))
    if "def _create_extractor(metric_config: dict):" in src:
        cur_src = src
        break

for cell in base_nb.get("cells", []):
    if cell.get("cell_type") != "code":
        continue
    src = "\n".join(cell.get("source", []))
    if "def extract_num_figures(" in src:
        base_src = src
        break

if cur_src is None or base_src is None:
    raise RuntimeError("Could not locate current/base num_figures source")

ns_cur = {"re": re, "fitz": fitz}
exec(cur_src, ns_cur)
extract_cur = ns_cur["extract_num_figures"]

ns_base = {"re": re, "fitz": fitz}
exec(base_src, ns_base)
extract_base = ns_base["extract_num_figures"]

raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")

rows = []
for _, row in validation[validation["num_figures"].notna()].iterrows():
    file_name = str(row["pdf_file"]).strip()
    pdf_path = raw_root / file_name
    if not pdf_path.exists():
        continue
    cur = extract_cur(str(pdf_path), debug=False)
    base = extract_base(str(pdf_path), debug=False)
    rows.append(
        {
            "pdf_file": file_name,
            "true_num_figures": row["num_figures"],
            "current_pred": cur,
            "base_pred": base,
            "delta": cur - base,
        }
    )

cmp_df = pd.DataFrame(rows)
print("num_figures current vs git-base")
print(f"- files checked: {len(cmp_df)}")
print(f"- files with deltas: {int((cmp_df['delta'] != 0).sum())}")
display(cmp_df.sort_values('delta', ascending=False))


## - LEGACY production cell

In [ ]:
# ==== FUNCTIONS (Element 1.2 & 1.4) ====
# Production-ready extractor plumbing for num_tables and num_references.
# num_figures is intentionally defined in the dedicated patch cell below.


def _create_extractor(metric_config: dict):
    """
    Factory function that creates an extractor function for table-like list detection.

    Args:
        metric_config: dict with keys:
            - token_pattern: regex for token (table terms)
            - index_pattern: regex for index numbering
            - heading_terms: tuple of heading terms
            - ignored_markers: tuple of heading markers to skip
            - opposite_heading_pattern: pattern for opposite list (figures in LOT)
            - standalone_idx_pattern: optional stricter pattern
            - metric_name: metric label for debug output
    """

    def extractor(pdf_path: str, debug: bool = False) -> int | None:
        try:
            token_pattern = metric_config["token_pattern"]
            arabic_index_pattern = metric_config["arabic_index_pattern"]
            roman_index_pattern = metric_config["roman_index_pattern"]
            index_pattern = metric_config["index_pattern"]
            heading_terms = metric_config["heading_terms"]
            ignored_markers = metric_config["ignored_markers"]
            opposite_heading_pattern = metric_config["opposite_heading_pattern"]
            standalone_idx_pattern = metric_config.get("standalone_idx_pattern")
            metric_name = metric_config["metric_name"]

            def _normalize_idx(idx_text: str) -> str:
                normalized = re.sub(r"\s+", "", idx_text.strip("() ")).replace(",", ".")
                return normalized.upper()

            def _is_toc_context(lines: list[str], heading_line_num: int) -> bool:
                start = max(0, heading_line_num - 8)
                context = " ".join(lines[start:heading_line_num]).lower()
                toc_markers = ("contents", "preface", "acknowledgements")
                return any(marker in context for marker in toc_markers)

            caption_start_pattern = re.compile(
                rf"^\s*(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\s*(?P<sep>[:\-\.,])?\s*(?P<tail>.*)$",
                re.IGNORECASE,
            )
            caption_inline_strong_pattern = re.compile(
                rf"(?<!\w)(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\s*(?P<sep>[:\-])\s*(?P<tail>.+)$",
                re.IGNORECASE,
            )
            token_presence_pattern = re.compile(
                rf"(?<!\w){token_pattern}(?!\w)",
                re.IGNORECASE,
            )
            numbered_heading_prefix_pattern = rf"(?:{roman_index_pattern}|[A-Z]|\d+(?:\s*[\.-]\s*\d+)*)"
            heading_pattern = re.compile(
                rf"^\s*(?:{numbered_heading_prefix_pattern}\s*[\).:-]?\s+)?(?:{'|'.join(re.escape(term) for term in heading_terms)})\s*:?\s*$",
                re.IGNORECASE,
            )
            entry_pattern = re.compile(
                rf"^\s*(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\b.*$",
                re.IGNORECASE,
            )
            idx_caption_pattern = re.compile(
                rf"^\s*(?P<idx>{index_pattern})\s+(?P<tail>.+)$",
                re.IGNORECASE,
            )
            caption_letters_pattern = re.compile(r"[A-Za-zÆØÅæøå]")
            generic_heading_pattern = re.compile(
                r"^\s*(chapter\s+\d+|kapitel\s+\d+|references|litteratur|bibliography|appendix|bilag)\b",
                re.IGNORECASE,
            )

            if standalone_idx_pattern:
                idx_pattern_compiled = re.compile(standalone_idx_pattern, re.IGNORECASE)
            else:
                idx_pattern_compiled = re.compile(
                    rf"^\s*(?P<idx>{index_pattern})\s*$",
                    re.IGNORECASE,
                )

            doc = fitz.open(pdf_path)
            pages_lines = []
            for page_num, page in enumerate(doc, start=1):
                page_text = page.get_text("text") or ""
                lines = [re.sub(r"\s+", " ", ln).strip() for ln in page_text.splitlines()]
                pages_lines.append((page_num, lines))

            heading_found_page = None
            for scan_page_num, scan_lines in pages_lines:
                for scan_line_num, scan_line in enumerate(scan_lines, start=1):
                    if not scan_line:
                        continue
                    if heading_pattern.match(scan_line):
                        if not _is_toc_context(scan_lines, scan_line_num - 1):
                            heading_found_page = scan_page_num
                            break
                if heading_found_page is not None:
                    break

            mode = False
            seen = False
            entries = set()
            first_numbers = set()
            debug_lines = []
            pending_idx = None
            pending_meta = None
            pending_first_num = None
            start_page = None
            max_span_pages = 12

            in_text_cues = (
                "see",
                "shown in",
                "illustrated in",
                "reported in",
                "as shown in",
                "as seen in",
                "som vist i",
                "se",
                "se også",
                "illustreret i",
                "consider",
                "at",
                "on",
                "again",
                ". ",
                "from",
            )
            no_sep_reference_tail_starters = ("for ", "in ", "of ", "from ")

            scan_start_idx = 0
            if heading_found_page is not None:
                scan_start_idx = next((i for i, (pn, _) in enumerate(pages_lines) if pn == heading_found_page), 0)

            for page_idx, (page_num, lines) in enumerate(pages_lines[scan_start_idx:], start=scan_start_idx):
                for line_num, line in enumerate(lines, start=1):
                    if not line:
                        continue

                    if heading_pattern.match(line):
                        if _is_toc_context(lines, line_num - 1):
                            if debug:
                                debug_lines.append((page_num, line_num, f"{metric_name}-heading-rejected-toc-context", line))
                            continue
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        mode = True
                        seen = True
                        start_page = page_num
                        if debug:
                            debug_lines.append((page_num, line_num, f"{metric_name}-heading", line))
                        continue

                    if not mode:
                        continue

                    if start_page is not None and page_num - start_page > max_span_pages:
                        if pending_idx is not None and debug:
                            p_page, p_line = pending_meta
                            debug_lines.append((p_page, p_line, f"{metric_name}-standalone-rejected-span", pending_idx))
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        mode = False
                        if debug:
                            debug_lines.append((page_num, line_num, f"{metric_name}-end-span", line))
                        continue

                    if opposite_heading_pattern.match(line):
                        if pending_idx is not None and debug:
                            p_page, p_line = pending_meta
                            debug_lines.append((p_page, p_line, f"{metric_name}-standalone-rejected-opposite-switch", pending_idx))
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        mode = False
                        if debug:
                            debug_lines.append((page_num, line_num, f"{metric_name}-end-opposite-heading", line))
                        continue

                    if generic_heading_pattern.match(line):
                        if pending_idx is not None and debug:
                            p_page, p_line = pending_meta
                            debug_lines.append((p_page, p_line, f"{metric_name}-standalone-rejected-no-caption", pending_idx))
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        mode = False
                        if debug:
                            debug_lines.append((page_num, line_num, f"{metric_name}-end-heading", line))
                        continue

                    entry_match = entry_pattern.match(line)
                    if entry_match:
                        idx_norm = _normalize_idx(entry_match.group("idx"))
                        entries.add(idx_norm)
                        first_num_match = re.search(r"\d+", idx_norm)
                        if first_num_match:
                            first_numbers.add(int(first_num_match.group()))
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        if debug:
                            debug_lines.append((page_num, line_num, f"{metric_name}-entry", line))
                        continue

                    idx_match = idx_pattern_compiled.match(line)
                    if idx_match:
                        idx_norm = _normalize_idx(idx_match.group("idx"))
                        if pending_idx is not None and debug:
                            p_page, p_line = pending_meta
                            debug_lines.append((p_page, p_line, f"{metric_name}-standalone-replaced", pending_idx))
                        pending_idx = idx_norm
                        pending_meta = (page_num, line_num)
                        first_num_match = re.search(r"\d+", idx_norm)
                        pending_first_num = int(first_num_match.group()) if first_num_match else None
                        if debug:
                            debug_lines.append((page_num, line_num, f"{metric_name}-standalone-candidate", line))
                        continue

                    idx_cap_match = idx_caption_pattern.match(line)
                    if idx_cap_match:
                        idx_norm = _normalize_idx(idx_cap_match.group("idx"))
                        tail = idx_cap_match.group("tail").strip()
                        tail_lower = tail.lower()
                        tail_word_count = len(tail.split())
                        tail_has_dot_leader = bool(re.search(r"\.{3,}", tail))
                        tail_has_page_number = bool(re.search(r"\b\d{1,4}\s*$", tail))

                        if (
                            caption_letters_pattern.search(tail)
                            and not opposite_heading_pattern.match(tail)
                            and not tail_lower.startswith(("references", "bibliography", "contents"))
                            and tail_word_count >= 2
                            and (tail_has_dot_leader or tail_has_page_number)
                        ):
                            entries.add(idx_norm)
                            first_num_match = re.search(r"\d+", idx_norm)
                            if first_num_match:
                                first_numbers.add(int(first_num_match.group()))
                            if debug:
                                debug_lines.append((page_num, line_num, f"{metric_name}-leading-idx-entry", line))
                            pending_idx = None
                            pending_meta = None
                            pending_first_num = None
                            continue

                    if pending_idx is not None and caption_letters_pattern.search(line):
                        caption_word_count = len(line.split())
                        is_heading_like = bool(heading_pattern.match(line) or generic_heading_pattern.match(line))
                        has_opposite_switch = bool(opposite_heading_pattern.match(line))
                        max_seen_first = max(first_numbers) if first_numbers else None
                        is_plausible_first_num = (
                            pending_first_num is not None
                            and pending_first_num <= 60
                            and (
                                max_seen_first is None
                                or pending_first_num <= max_seen_first + 5
                                or pending_first_num in first_numbers
                            )
                        )

                        if is_heading_like or has_opposite_switch or caption_word_count < 2 or not is_plausible_first_num:
                            if debug:
                                debug_lines.append((page_num, line_num, f"{metric_name}-standalone-rejected-caption-check", line))
                            continue

                        entries.add(pending_idx)
                        first_numbers.add(pending_first_num)
                        if debug:
                            p_page, p_line = pending_meta
                            debug_lines.append((p_page, p_line, f"{metric_name}-standalone-accepted", pending_idx))
                            debug_lines.append((page_num, line_num, f"{metric_name}-caption-for-standalone", line))
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        continue

                    if debug and token_presence_pattern.search(line):
                        debug_lines.append((page_num, line_num, f"{metric_name}-non-entry", line))

            if pending_idx is not None and debug:
                p_page, p_line = pending_meta
                debug_lines.append((p_page, p_line, f"{metric_name}-standalone-rejected-eof", pending_idx))

            if seen and len(entries) >= 4:
                if debug:
                    print(f"[DEBUG {metric_name}] {pdf_path}")
                    print("  mode=fast-track-list")
                    print(f"  entries={len(entries)}")
                    for p, l, reason, text in debug_lines[:120]:
                        print(f"  + p{p}:l{l} [{reason}] {text}")
                    if len(debug_lines) > 120:
                        print(f"  ... {len(debug_lines) - 120} more lines omitted")
                doc.close()
                return len(entries)

            unique_keys = set()
            accepted_debug = []
            rejected_debug = []

            for page_num, lines in pages_lines:
                for line_num, line in enumerate(lines, start=1):
                    if not line:
                        continue

                    lower_line = line.lower()
                    has_token = bool(token_presence_pattern.search(line))

                    if any(marker in lower_line for marker in ignored_markers):
                        if debug and has_token:
                            rejected_debug.append((page_num, line_num, "heading/list section", line))
                        continue

                    if len(line) > 220:
                        if debug and has_token:
                            rejected_debug.append((page_num, line_num, "too long for caption", line))
                        continue

                    match_start = caption_start_pattern.match(line)
                    if match_start:
                        sep = (match_start.group("sep") or "").strip()
                        tail = (match_start.group("tail") or "").strip().lower()
                        if not sep and any(tail.startswith(starter) for starter in no_sep_reference_tail_starters):
                            if debug:
                                rejected_debug.append((page_num, line_num, "start-line reference fragment (no separator)", line))
                            continue

                        idx_raw = match_start.group("idx")
                        idx_compact = re.sub(r"\s+", "", idx_raw.strip("() "))
                        if not sep and idx_compact and idx_compact[-1].isalpha() and tail:
                            split_joined_tail = (idx_compact[-1] + tail).lower()
                            if any(split_joined_tail.startswith(starter.strip()) for starter in no_sep_reference_tail_starters):
                                if debug:
                                    rejected_debug.append((page_num, line_num, "start-line reference fragment (split token)", line))
                                continue

                        idx_norm = _normalize_idx(idx_raw)
                        key = (page_num, idx_norm)
                        unique_keys.add(key)
                        if debug:
                            accepted_debug.append((page_num, line_num, "start-of-line caption", line))
                        continue

                    match_inline = caption_inline_strong_pattern.search(line)
                    if match_inline:
                        prefix = line[: match_inline.start()].lower()
                        if any(cue in prefix for cue in in_text_cues):
                            if debug:
                                rejected_debug.append((page_num, line_num, "in-text reference cue", line))
                            continue

                        if match_inline.start() <= 20 and len(line) <= 160:
                            idx_raw = match_inline.group("idx")
                            idx_norm = _normalize_idx(idx_raw)
                            key = (page_num, idx_norm)
                            unique_keys.add(key)
                            if debug:
                                accepted_debug.append((page_num, line_num, "inline strong fallback", line))
                            continue

                        if debug:
                            rejected_debug.append((page_num, line_num, "weak inline candidate", line))
                        continue

                    if debug and has_token:
                        rejected_debug.append((page_num, line_num, "token seen but no valid caption format", line))

            doc.close()

            if debug:
                print(f"[DEBUG {metric_name}] {pdf_path}")
                print("  mode=fallback-caption-scan")
                if seen and entries:
                    print("  note=List heading found but entries were too uncertain; fallback used")
                print(f"  accepted_candidates={len(accepted_debug)} unique_count={len(unique_keys)}")
                for p, l, reason, text in accepted_debug[:60]:
                    print(f"  + p{p}:l{l} [{reason}] {text}")
                if len(accepted_debug) > 60:
                    print(f"  ... {len(accepted_debug) - 60} more accepted lines omitted")
                print(f"  rejected_candidates={len(rejected_debug)}")
                for p, l, reason, text in rejected_debug[:80]:
                    print(f"  - p{p}:l{l} [{reason}] {text}")
                if len(rejected_debug) > 80:
                    print(f"  ... {len(rejected_debug) - 80} more rejected lines omitted")

            return len(unique_keys)

        except Exception as e:
            print(f"[{metric_config['metric_name']}] Failed for {pdf_path}: {e}")
            return None

    return extractor


# ============================================================================
# TABLES EXTRACTOR CONFIGURATION
# ============================================================================
_ARABIC_IDX_PATTERN = r"(?:[A-Z]\s*[\.-]\s*)?\d+(?:\s*[\.,-]\s*\d+)*(?:\s*[a-zA-Z])?"
_ROMAN_IDX_PATTERN = r"[IVXLCDM]{1,7}"
_INDEX_PATTERN = rf"\(?\s*(?:{_ARABIC_IDX_PATTERN}|{_ROMAN_IDX_PATTERN})\s*\)?"

# Kept for table parser boundary detection (List-of-Figures switch).
_FIG_HEADINGS = (
    "list of figures",
    "figure list",
    "lof",
    "figures",
    "figurer",
    "figurenliste",
    "figur liste",
    "liste over figurer",
    "figuroversigt",
    "oversigt over figurer",
    "fortegnelse over figurer",
    "liste af figurer",
)

_TABLE_HEADINGS = (
    "list of tables",
    "table list",
    "lot",
    "tables",
    "tabeller",
    "tabeloversigt",
    "oversigt over tabeller",
    "fortegnelse over tabeller",
    "liste over tabeller",
    "liste af tabeller",
)

_tab_config = {
    "token_pattern": r"(?:table|tab\.?|tabel|t\s*a\s*b(?:\s*l(?:\s*e)?)?\.?)",
    "arabic_index_pattern": _ARABIC_IDX_PATTERN,
    "roman_index_pattern": _ROMAN_IDX_PATTERN,
    "index_pattern": _INDEX_PATTERN,
    "heading_terms": _TABLE_HEADINGS,
    "ignored_markers": _TABLE_HEADINGS,
    "opposite_heading_pattern": re.compile(
        rf"^\s*(?:{_ROMAN_IDX_PATTERN}|[A-Z]|\d+(?:\s*[\.-]\s*\d+)*)\s*[\).:-]?\s+(?:{'|'.join(re.escape(t) for t in _FIG_HEADINGS)})\s*:?\s*$",
        re.IGNORECASE,
    ),
    "standalone_idx_pattern": r"^\s*(?P<idx>\d+(?:\s*[\.,-]\s*\d+)*(?:\s*[a-zA-Z])?)\s*$",
    "metric_name": "num_tables",
}

extract_num_tables = _create_extractor(_tab_config)


# For num_tables, keep metric-specific tuned logic from the archived cell.
def _load_archived_num_tables():
    try:
        import json
        from pathlib import Path

        nb_path = Path("thesis_stats_extractor.ipynb")
        if not nb_path.exists():
            nb_path = Path("exstraction_more_from_pdf/thesis_stats_extractor.ipynb")

        nb = json.loads(nb_path.read_text())
        archived_src = None
        for c in nb.get("cells", []):
            if c.get("cell_type") != "code":
                continue
            src = "\n".join(c.get("source", []))
            if (
                "def extract_num_tables(pdf_path: str, debug: bool = False)" in src
                and "def _create_extractor(metric_config: dict):" not in src
            ):
                archived_src = src
                break

        if archived_src is None:
            return None

        ns = {"re": re, "fitz": fitz}
        exec(archived_src, ns)
        return ns.get("extract_num_tables")
    except Exception:
        return None


_archived_num_tables = _load_archived_num_tables()
if callable(_archived_num_tables):
    extract_num_tables = _archived_num_tables


# For num_references, keep metric-specific tuned logic from the development cell.
# Only the function block is loaded so local test-run code is not executed.
def _load_archived_num_references():
    try:
        import json
        from pathlib import Path

        nb_path = Path("thesis_stats_extractor.ipynb")
        if not nb_path.exists():
            nb_path = Path("exstraction_more_from_pdf/thesis_stats_extractor.ipynb")

        nb = json.loads(nb_path.read_text())
        archived_src = None
        for c in nb.get("cells", []):
            if c.get("cell_type") != "code":
                continue
            source_lines = c.get("source", [])
            first_nonempty = next((ln.strip() for ln in source_lines if ln.strip()), "")
            src = "\n".join(source_lines)
            if (
                first_nonempty == "# ==== DEVELOPMENT (Element 1.4: num_references) ===="
                and "def extract_num_references(pdf_path: str, debug: bool = False)" in src
            ):
                archived_src = src
                break

        if archived_src is None:
            return None

        lines = archived_src.splitlines()
        start_idx = None
        for i, line in enumerate(lines):
            if line.startswith("def extract_num_references("):
                start_idx = i
                break

        if start_idx is None:
            return None

        end_idx = len(lines)
        marker = "# Local-only test run for metric 1.4: num_references"
        for i in range(start_idx + 1, len(lines)):
            if marker in lines[i]:
                end_idx = i
                break

        func_src = "\n".join(lines[start_idx:end_idx]).rstrip()
        if not func_src:
            return None

        ns = {"re": re, "fitz": fitz}
        exec(func_src, ns)
        return ns.get("extract_num_references")
    except Exception:
        return None


_archived_num_references = _load_archived_num_references()
if callable(_archived_num_references):
    extract_num_references = _archived_num_references
else:

    def extract_num_references(pdf_path: str, debug: bool = False) -> int | None:
        """Hard-fail fallback to prevent silent degradation in production runs."""
        raise RuntimeError(
            "num_references extractor failed to load from DEVELOPMENT cell. Run DEVELOPMENT cell once or fix loader selector."
        )